# Gut bacteriome and mycobiome in glioblastoma

Analysis code for:

> Herz M, Schmid K, Ankenbrand M, et al. Exploring the gut-brain axis in glioblastoma: a prospective case-control study of the gut bacteriome and mycobiome.

The notebook reproduces all statistical analyses, tables and figures of the manuscript, based on 16S rRNA gene (genus level) and ITS (species level) sequencing of rectal swabs from 23 patients with newly diagnosed IDH-wildtype glioblastoma and 23 healthy controls.

## Contents

1. Import and filtering of the 16S and ITS datasets (samples with ≥ 1000 reads)
2. Clinical metadata, serum cytokines and glia markers; anonymization
3. Descriptive statistics (Table 1, Table 2)
4. Survival analysis: Kaplan-Meier estimate, Cox regression with model diagnostics and sensitivity analyses
5. Microbiome analyses for patients vs. controls and for patients by overall survival (split at the cohort median):
   alpha diversity (DivNet, `testDiversity`), beta diversity (Bray-Curtis, `testBetaDiversity`), differential abundance (radEmu)
6. Composite main figures (Figures 2–4) and supplementary figures (Figures S1–S3)

## Data availability

Raw sequencing data are available from the European Nucleotide Archive under accession PRJEB91343. The file `sample_mapping.csv` assigns each sequencing library to its study group and, for patients, to the overall survival group. The column participant corresponds to the sample alias in ENA (PRJEB91343); run aliases follow the pattern webin-reads-<participant>_<marker>. Libraries not listed were excluded during quality filtering (< 1,000 reads). Clinical metadata cannot be shared publicly for data protection reasons and are available from the corresponding author on reasonable request.

Expected input structure:

    data/
    ├── ITS/    taxonomy.vsearch, asv_table.merge.txt, samples.csv
    ├── 16S/    taxonomy.vsearch, asv_table.merge.txt, samples.csv
    ├── metadataGBM_v2.csv, Medikamente_v2.csv
    ├── Serumproben_umformatiert.csv, Microglia_Darm-Hirn.csv
    └── iba1_case_*.tif

ASV and taxonomy tables were generated with the `metabarcoding_pipeline` by Alexander Keller (https://github.com/chiras/metabarcoding_pipeline); the helper functions in the Setup section are taken from this pipeline.

## Requirements

R 4.4.1 with the IRkernel. CRAN packages are loaded in the Setup section; `conflicted` must also be installed. DivNet (v0.4.1) and radEmu (v2.2.1.0) were installed from GitHub.

    remotes::install_github("adw96/DivNet")
    remotes::install_github("statdivlab/radEmu")
    BiocManager::install("phyloseq", version = "3.19")

Exact package versions are listed in `sessionInfo.txt`.

## Notes

- All outputs are written to `output_glioblastoma/`, which is created on first run.
- DivNet and radEmu fits are computationally intensive; random seeds are set before each DivNet fit.
- The group labels used internally (`c`/`p`, `lowerOS`/`higherOS`) are mapped to display labels (Controls/Patients, Shorter OS/Longer OS) in the Setup section.

## License

MIT, see `LICENSE`.  `sample_mapping.csv` is released under CC BY 4.0.
Helper functions in the Setup section are adapted from metabarcoding_pipeline by Alexander Keller (MIT License).

# Setup

In [ ]:
library(phyloseq)
library(tidyverse)
library(RColorBrewer)
library(DivNet)
library(lubridate)
library(MASS)
library(magrittr)
library(radEmu)
library(survival)
library(survminer)
library(dendextend)
library(patchwork)
library(foreach)
library(broom)
library(gridExtra)
library(coin)
library(car)
library(corrplot)
library(vegan)   
library(ape)     
library(doParallel)
library(doSNOW)
library(ggh4x)
library(png)
library(grid)
library(tiff)


In [ ]:
conflicted::conflicts_prefer(dplyr::filter, dplyr::select, dplyr::lag, dplyr::recode)

Helper functions in the Setup section are adapted from metabarcoding_pipeline by Alexander Keller (MIT License).

In [ ]:
# Propagate taxonomy for tax_glom to work
propagate_incomplete_taxonomy <- function(phyloseq){
	taxranks <- colnames(tax_table(phyloseq))
	for (i in 2:length(taxranks)){
		tax_table(phyloseq)[is.na(tax_table(phyloseq)[,taxranks[i]]),taxranks[i]]<-paste(tax_table(phyloseq)[is.na(tax_table(phyloseq)[,taxranks[i]]),taxranks[i-1]],"_spc",sep="")
	}
	  return(phyloseq)
}


In [ ]:

replace_tax_prefixes <- function(phyloseq){
  tmp_tax_table <- apply(tax_table(phyloseq), c(1, 2),function(y) gsub("^.*\\.[a-z]:","",y))
  tmp_tax_table <- apply(tmp_tax_table,       c(1, 2),function(y) gsub("^\\w:","",y))
  tmp_tax_table <- apply(tmp_tax_table,       c(1, 2),function(y) gsub("_spc_.*","_spc",y))
  tmp_tax_table <- apply(tmp_tax_table,       c(1, 2),function(y) gsub("_"," ",y))
  tmp_tax_table <- apply(tmp_tax_table,       c(1, 2),function(y) gsub(";$","",y))
  tax_table(phyloseq)<- tmp_tax_table
  return(phyloseq)
}

In [ ]:
max(1,n_cores <- parallel::detectCores() - 2)

In [ ]:
theme_pub <- function(base_size = 8) {
  theme_minimal(base_size = base_size) +
    theme(
      panel.grid.minor = element_blank(),
      legend.position  = "bottom",
      legend.key.size  = unit(0.3, "cm"),
      plot.title       = element_text(face = "bold"),
      strip.text       = element_text(face = "bold")
    )
}


# Group palettes: same group == same colour in EVERY figure
col_pvc     <- c("Controls" = "#056b05", "Patients" = "#45458b")   # recoded labels
col_os      <- c("Shorter OS" = "#550d99", "Longer OS" = "#0e57a0")
col_pvc_raw <- c("control"  = "#056b05", "patient"   = "#45458b")  # raw factor levels
col_os_raw  <- c("lowerOS"  = "#550d99", "higherOS"  = "#0e57a0")

# Display labels for raw factor levels and cytokine columns
pvc_labels <- c(control = "Controls", patient = "Patients")
os_labels  <- c(lowerOS = "Shorter OS", higherOS = "Longer OS")
il_labels  <- c(IFN_alpha = "IFN-\u03b1", IL_10 = "IL-10", IL_17AF = "IL-17A/F",
                IL_2 = "IL-2", IL_4 = "IL-4", IL_6 = "IL-6", TNF_alpha = "TNF-\u03b1")

# Journal widths (cm) + one save helper for all figures
W_SINGLE <- 8.5;  W_DOUBLE <- 17.5

save_fig <- function(filename, plot, width = W_SINGLE, height = 8.5,
                     units = "cm", dpi = 300, bg = "white", ...) {
  if (is.null(plot)) return(invisible(NULL))   # skip empty plots (e.g. no significant taxa)
  ggsave(filename = filename, plot = plot, width = width, height = height,
         units = units, dpi = dpi, bg = bg, ...)
}

theme_set(theme_pub())   # global default for every plot

# Colour palette for taxa in barplots
expandedPalette <- colorRampPalette(brewer.pal(9, "Paired"))

In [ ]:
dir.create("output_glioblastoma", showWarnings = FALSE)

# Import data

## ITS

In [ ]:
path = "path/to/metabarcoding_pipeline/results/ITS"

In [ ]:
taxtab <- read_csv(str_glue("{path}/taxonomy.vsearch"))

In [ ]:
taxa <- as.matrix(taxtab[,2:ncol(taxtab)])
row.names(taxa) <- pull(taxtab[,1])

In [ ]:
otutab <- read_tsv(str_glue("{path}/asv_table.merge.txt"))

In [ ]:
otus <- as.matrix(otutab[,2:ncol(otutab)])
row.names(otus) <- pull(otutab[,1])

In [ ]:
samples <- read_csv2(str_glue("{path}/samples.csv")) %>% mutate_all(as.factor) %>% as.data.frame
rownames(samples) <- samples$name

In [ ]:
samples <- samples %>% transmute(project,name,sample=str_remove(name, "fITS7-ITS4-P01-...-."), type=str_sub(name,-4,-4))

In [ ]:
phy_ITS_raw <- phyloseq(otu_table(otus, taxa_are_rows = T), tax_table(taxa), sample_data(samples))
phy_ITS_raw

In [ ]:
sample_data(phy_ITS_raw) %>% as_tibble 

## 16S

In [ ]:
path="path/to/metabarcoding_pipeline/results/16S"

In [ ]:
taxtab <- read_csv(str_glue("{path}/taxonomy.vsearch"))

In [ ]:
taxa <- as.matrix(taxtab[,2:ncol(taxtab)])
row.names(taxa) <- pull(taxtab[,1])

In [ ]:
otutab <- read_tsv(str_glue("{path}/asv_table.merge.txt"))

In [ ]:
otus <- as.matrix(otutab[,2:ncol(otutab)])
row.names(otus) <- pull(otutab[,1])

In [ ]:
samples <- read_csv2(str_glue("{path}/samples.csv")) %>% mutate_all(as.factor) %>% as.data.frame
rownames(samples) <- samples$name

In [ ]:
samples <- samples %>% transmute(project,name,sample=str_remove(name, "341F-785R-P01-...-."), type=str_sub(name,-4,-4))

In [ ]:
phy_16S_raw <- phyloseq(otu_table(otus, taxa_are_rows = T), tax_table(taxa), sample_data(samples))

In [ ]:
phy_16S_raw

### Explore sequencing controls

In [ ]:
summary(otu_table(phy_ITS_raw)[,c("fITS7-ITS4-P01-H06-Buffer-empty","fITS7-ITS4-P01-F06-Extcontrol-empty")])

In [ ]:
summary(otu_table(phy_16S_raw)[,c("341F-785R-P01-F06-Extcontrol-empty","341F-785R-P01-H06-Buffer-empty" )])

In [ ]:
tibble(
  sample = names(sample_sums(phy_ITS_raw)),
  reads  = sample_sums(phy_ITS_raw)
) %>%
  arrange(desc(reads))

# Filtered dataset

## Filter samples 
with less than 1000 reads to obtain reliable results

### ITS

In [ ]:
phy_ITS_raw_filtered <- prune_samples(sample_sums(phy_ITS_raw) >= 1000, phy_ITS_raw)
phy_ITS_raw_filtered <- prune_samples(sample_data(phy_ITS_raw_filtered)$type %in% c("c","p"), phy_ITS_raw_filtered)
phy_ITS_raw_filtered

In [ ]:
# Number of samples before filtering
table(sample_data(phy_ITS_raw)$type)

In [ ]:
# Number of samples passing filters per group
table(sample_data(phy_ITS_raw_filtered)$type)

### 16S

In [ ]:
phy_16S_raw_filtered <- prune_samples(sample_sums(phy_16S_raw) >= 1000, phy_16S_raw)
phy_16S_raw_filtered <- prune_samples(sample_data(phy_16S_raw_filtered)$type %in% c("c", "p"), phy_16S_raw_filtered)
phy_16S_raw_filtered

In [ ]:
# Number of samples before filtering
table(sample_data(phy_16S_raw)$type)

In [ ]:
# Number of samples passing filters per group
table(sample_data(phy_16S_raw_filtered)$type)

## Aggregate at genus/species level

### ITS

In [ ]:
phy_ITS_spc_filtered <- phy_ITS_raw_filtered %>%      
  propagate_incomplete_taxonomy() %>%
  replace_tax_prefixes() %>%
  tax_glom("species", NArm = FALSE) %>%
  prune_taxa(taxa_sums(.) > 0, .)

phy_ITS_spc_filtered

In [ ]:
# Relative abundance 
phy_ITS_rel_spc_filtered <- transform_sample_counts(
  phy_ITS_spc_filtered, function(x) x / sum(x)
)
phy_ITS_rel_spc_filtered

### 16S

In [ ]:
phy_16S_gen_filtered <- phy_16S_raw_filtered %>%       
  propagate_incomplete_taxonomy() %>%
  replace_tax_prefixes() %>%
  tax_glom("genus", NArm = FALSE) %>%
  prune_taxa(taxa_sums(.) > 0, .)

phy_16S_gen_filtered

In [ ]:
# Relative abundance
phy_16S_rel_gen_filtered <- transform_sample_counts(
  phy_16S_gen_filtered, function(x) x / sum(x)
)
phy_16S_rel_gen_filtered

In [ ]:
summary(sample_sums(phy_16S_gen_filtered))
summary(sample_sums(phy_ITS_spc_filtered))

In [ ]:
sum(sample_sums(phy_16S_gen_filtered))
sum(sample_sums(phy_ITS_spc_filtered))

### Number and fraction of counts not classified at species/genus level

In [ ]:
# Fraction of reads classified at species and genus level (ITS)
tax_tab_ITS <- as.data.frame(tax_table(phy_ITS_raw_filtered))
read_per_taxon <- taxa_sums(phy_ITS_raw_filtered)

tibble(
  taxon = names(read_per_taxon),
  reads = read_per_taxon,
  phylum_classified  = !is.na(tax_tab_ITS$phylum),
  genus_classified   = !is.na(tax_tab_ITS$genus),
  species_classified = !is.na(tax_tab_ITS$species)
) %>%
  summarise(
    total_reads          = sum(reads),
    reads_with_phylum    = sum(reads[phylum_classified]),
    reads_with_genus     = sum(reads[genus_classified]),
    reads_with_species   = sum(reads[species_classified]),
    prop_phylum          = reads_with_phylum  / total_reads,
    prop_genus           = reads_with_genus   / total_reads,
    prop_species         = reads_with_species / total_reads
  )

In [ ]:
# Fraction of reads classified at each level (16S)
# Direct approach without psmelt — uses tax_table and otu_table directly
tax_tab_16S <- as.data.frame(tax_table(phy_16S_raw_filtered))
read_per_taxon <- taxa_sums(phy_16S_raw_filtered)

tibble(
  taxon = names(read_per_taxon),
  reads = read_per_taxon,
  family_classified = !is.na(tax_tab_16S$family),
  genus_classified  = !is.na(tax_tab_16S$genus)
) %>%
  summarise(
    total_reads        = sum(reads),
    reads_with_family  = sum(reads[family_classified]),
    reads_with_genus   = sum(reads[genus_classified]),
    prop_family        = reads_with_family / total_reads,
    prop_genus         = reads_with_genus  / total_reads
  )

# Metadata

## Set up

In [ ]:
path_meta = "path/to/metadata/" 

In [ ]:
drugs<-read_csv2(str_glue("{path_meta}Medikamente_v2.csv"))

In [ ]:
meta<-read_csv2(str_glue("{path_meta}metadataGBM_v2.csv"))

In [ ]:
drugs<-drugs %>%
  mutate(
    Pseudonymisierung = as.character(Pseudonymisierung),
    Status = factor(Status, levels = c(0,1), labels = c("patient", "control")),
    Sex = factor(Sex, levels = c(1,2), labels = c("male", "female")),
    medication = factor(medication, levels = c(0,1), labels = c("no", "yes")),
    antihypertensives = factor(antihypertensives, levels = c(0,1), labels = c("no", "yes")),
    anticoagulants = factor(anticoagulants, levels = c(0,1), labels = c("no", "yes")),
    gastric_protection = factor(gastric_protection, levels = c(0,1), labels = c("no", "yes")),
    analgesics = factor(analgesics, levels = c(0,1), labels = c("no", "yes")),
    antidiabetics = factor(antidiabetics, levels = c(0,1), labels = c("no", "yes")),
    sedatives = factor(sedatives, levels = c(0,1), labels = c("no", "yes")),
    statins = factor(statins, levels = c(0,1), labels = c("no", "yes")),
    prostate = factor(prostate, levels = c(0,1), labels = c("no", "yes")),
    hormonal = factor(hormonal, levels = c(0,1), labels = c("no", "yes")),
    antiepileptics = factor(antiepileptics, levels = c(0,1), labels = c("no", "yes")),
    corticosteroids = factor(corticosteroids, levels = c(0,1), labels = c("no", "yes")),
    thyroid = factor(thyroid, levels = c(0,1), labels = c("no", "yes")),
    antidepressants = factor(antidepressants, levels = c(0,1), labels = c("no", "yes"))
    ) %>%
  rename(
    ID = Pseudonymisierung,
    status2 = Status,
    sex2 = Sex
  )


In [ ]:
# adjust data types and variable encoding 
meta<- meta %>%
  mutate(
    birth_date = as.Date(birth_date, format = "%d.%m.%Y"),
    ID = as.character(ID),
    status = factor(status, levels = c(1,0), labels = c("control", "patient")),
    sex = factor(sex, levels = c(1,2), labels = c("male", "female")),
    age = as.numeric(age),
    vaccination = factor(vaccination, levels = c(0,1), labels = c("no", "yes")),
    antibiotics = factor(antibiotics, levels = c(0,1), labels = c("no", "yes")),
    infect = factor(infect, levels = c(0,1), labels = c("no", "yes")),
    intestinal_disease = factor(intestinal_disease, levels = c(0,1), labels = c("no", "yes")),
    intestinal_disease_which = as.character(intestinal_disease_which),
    rheumatol_disease = factor(rheumatol_disease, levels = c(0,1), labels = c("no", "yes")),
    autoimmune_disease = factor(autoimmune_disease, levels = c(0,1), labels = c("no", "yes")),
    cancer = factor(cancer, levels = c(0,1), labels = c("no", "yes")),
    pregnancy = factor(pregnancy, levels = c(0,1), labels = c("no", "yes")),
    smoking = factor(smoking, levels = c(0,1), labels = c("no", "yes")),
    cigarettes_per_day = as.numeric(cigarettes_per_day),
    #alcohol_consumption = factor(alcohol_consumption, levels = 0:4, labels = c("none", "irregular", "several per month", "several per week", "daily")),
    drink_kind = as.character(drink_kind),
    drink_amount = as.numeric(drink_amount),
    # food_allergies = factor(food_allergies, levels = c(0,1), labels = c("no", "yes")),
    meat_consumption = factor (meat_consumption, levels = 0:4, labels = c("none", "irregular", "several per month", "several per week", "daily")),
    carbohydrates_consumption = factor(carbohydrates_consumption, levels = 0:4, labels = c("none", "irregular", "several per month", "several per week", "daily")),
    plants_consumption = factor(plants_consumption, levels = 0:4, labels = c("none", "irregular", "several per month", "several per week", "daily")),
    food_allergies_kind = as.character(food_allergies_kind),
    diet = factor(diet,  levels = c(0,1), labels = c("no", "yes")),
    diet_kind = factor(diet_kind, levels = 0:2, labels = c("vegetarian", "vegan", "other")),
    diet_kind_detail = as.character(diet_kind_detail),
    probiotics = factor(probiotics,  levels = c(0,1), labels = c("no", "yes")),
    probiotics_kind = factor(probiotics_kind, levels = 0:3, labels = c("yogurt", "kefir", "capsules", "other")),
    probiotics_frequency = factor(probiotics_frequency, levels = 0:4, labels = c("none", "irregular", "several per month", "several per week", "daily")),
    drugs = as.character(drugs),
    heart_failure = factor(heart_failure, levels = 0:2, labels = c("no", "NYHA I/II", "NYHA III/IV")),
    kidney_failure = factor (kidney_failure, levels = 0:2, labels = c("no", "Grade 1-2", "Grade 4/5")),
    liver_failure = factor(liver_failure, levels = 0:2, labels = c("no", "Child-Pugh A", "Child-Pugh B/C")),
    diabetes = factor(diabetes, levels = 0:2, labels = c("no", "not insulin dependent", "insulin dependent")),
    CREA = as.numeric(CREA),
    GFR = as.numeric(GFR),
    GOT = as.numeric(GOT),
    CRP = as.numeric(CRP),
    blood_glucose = as.numeric(blood_glucose),
    GPT = as.numeric(GPT),
    date_of_admission = as.Date(date_of_admission, format = "%d.%m.%Y"),
    #event_status_death = factor(event_status_death, levels = 0:1, labels = c("censored", "death")),
    end_date = as.Date(end_date, format = "%d.%m.%Y"),
    karnofsky = as.numeric(karnofsky),
    date_of_diagnosis = as.Date(date_of_diagnosis, format = "%d.%m.%Y"),
    diagnosis = as.character(diagnosis),
    localisation = as.character(localisation),
    date_of_surgery = as.Date(date_of_surgery, format = "%d.%m.%Y"),
    resection = as.factor(resection),
    patho_number = as.character(patho_number),
    ki67 = as.numeric(ki67),
    methylation = factor(methylation, levels = c(0,1), labels = c("no", "yes")),
    methylation_percentage = as.numeric(methylation_percentage),
    IDH1 = as.factor(IDH1),
      IDH2 = as.factor(IDH2),
    ATRX = as.factor(ATRX),
    radiation = factor(radiation,levels = c(0,1), labels = c("no", "yes")),
    radiation_dosage = as.numeric(radiation_dosage),
    start_of_radiation = as.Date(start_of_radiation, format = "%d.%m.%Y"),
    chemotherapy = factor(chemotherapy, levels = c(0,1), labels = c("no", "yes")),
    chemotherapy_start = as.Date(chemotherapy_start, format = "%d.%m.%Y"),
    chemotherapy_drug = factor(chemotherapy_drug, levels = 0:1, labels = c("temozolomide", "CeTeG")),
    chemotherapy_dosage = as.numeric(chemotherapy_dosage),
    chemotherapy_cycle_numbers = as.integer(chemotherapy_cycle_numbers),
    TTF = factor(TTF, levels = c(0,1), labels = c("no", "yes")),
    other_therapy = factor(other_therapy, levels = c(0,1), labels = c("no", "yes")),
    other_therapy_kind = as.character(other_therapy_kind),
    #event_status_relapse = factor(event_status_relapse, levels = 0:2, labels = c("censored", "relapse", "death before relapse")),
    date_of_relapse = as.Date(date_of_relapse,format = "%d.%m.%Y"),
    blood_sample = factor(blood_sample, levels = c(0,1), labels = c("no", "yes")),
    blood_date = as.Date(blood_date, format = "%d.%m.%Y"),
    serum_sample = factor(serum_sample, levels = c(0,1), labels = c("no", "yes")),
    serum_date = as.Date(serum_date, format = "%d.%m.%Y"),
    stool_sample = factor(stool_sample, levels = c(0,1), labels = c("no", "yes")),
    stool_date = as.Date(stool_date, format = "%d.%m.%Y")
  )


In [ ]:
meta <- meta %>%
  left_join(drugs, by = "ID")

In [ ]:
#check if status and sex are identical in meta and drugs and delete if they are
if (identical(meta$status, meta$status2)) {
  meta$status2 <- NULL
}
if (identical(meta$sex, meta$sex2)) {
  meta$sex2 <- NULL
}


In [ ]:
complete_meta<-meta

In [ ]:
write.csv(meta, str_glue("{path_meta}MetadataGBM_v3.csv"), row.names = FALSE)

## Calculate relative dates
Calculate all dates as time from diagnosis date

In [ ]:
(complete_meta)$event_status_death

In [ ]:
date_columns <- c("birth_date", "date_of_admission", "end_date", "date_of_diagnosis", 
                  "date_of_surgery", "start_of_radiation", "chemotherapy_start", 
                  "date_of_relapse", "blood_date", "serum_date", "stool_date")



# Compute differences in days from "date_of_diagnosis"
for (col in date_columns) {
  if (col == "birth_date") {
    complete_meta$age_at_diagnosis <- as.numeric(difftime(complete_meta$date_of_diagnosis, complete_meta$birth_date, units="days")) / 365.25
  } else if (col != "date_of_diagnosis") {
    new_col_name <- paste0("days_from_diagnosis_to_", col)
    complete_meta[[new_col_name]] <- as.numeric((complete_meta[[col]]- complete_meta$date_of_diagnosis))
  }
}

complete_meta$overall_survival <- ifelse(
  complete_meta$event_status_death == 1,
  complete_meta$days_from_diagnosis_to_end_date / 30.44,
  NA_real_
)


## Calculate alcohol consumption and number of cigarettes per day

#### alcohol consumption

The frequency and quantity data as well as the standard values for the mean alcohol contents of the beverages per litre – beer 38.11g, non-alcoholic beer 3.97g, wine 87.34g, distilled spirits 262.02g and cocktails/mixed drinks 75g – were used to estimate the average amount of alcohol in gram per day with the following formula (Lange2016)

In [ ]:
complete_meta$alcohol_consumption

In [ ]:
complete_meta$alcohol_g_day

In [ ]:
alcohol_content <- list(
  "wine" = 87.34,          
  "beer" = 38.11,            
  "wine spritzer" = 87.34/2,
  "german schnaps" = 262.02, 
  "gin" = 262.02             
)

# Mapping alcohol consumption frequency to days per 4 weeks
frequency_to_days <- c(0, 4, 8, 16, 28)  # 4 weeks
daily_conversion <- frequency_to_days / 28  # Convert to daily fraction


calculate_alcohol_g_day <- function(drink_kind, drink_amount, drink_kind2, drink_amount2, frequency) {
  # Ensure drink names are lowercase for correct matching
  drink_kind <- tolower(drink_kind)
  drink_kind2 <- tolower(drink_kind2)

  # If drink amounts are empty (NA), set them to 0
  drink_amount <- ifelse(is.na(drink_amount) | drink_amount == "", 0, as.numeric(drink_amount))
  drink_amount2 <- ifelse(is.na(drink_amount2) | drink_amount2 == "", 0, as.numeric(drink_amount2))

  # Get alcohol content for the drinks, default to 0 if empty
  alc_g_l_1 <- ifelse(drink_kind == "" | is.na(drink_kind), 0,
                      ifelse(drink_kind %in% names(alcohol_content), alcohol_content[[drink_kind]], {
                        warning(paste("Unbekanntes Getränk 1:", drink_kind))
                        0
                      }))

  alc_g_l_2 <- ifelse(drink_kind2 == "" | is.na(drink_kind2), 0,
                      ifelse(drink_kind2 %in% names(alcohol_content), alcohol_content[[drink_kind2]], {
                        warning(paste("Unbekanntes Getränk 2:", drink_kind2))
                        0
                      }))


  # Compute alcohol in grams for each drink
  alcohol_g_1 <- drink_amount * (alc_g_l_1 / 1000)  # Convert mL to L
  alcohol_g_2 <- drink_amount2 * (alc_g_l_2 / 1000) # Convert mL to L

  # Total alcohol per consumption event
  total_alcohol_g <- alcohol_g_1 + alcohol_g_2

  # Convert to daily intake
  alcohol_g_day <- total_alcohol_g * daily_conversion[frequency + 1]

  return(alcohol_g_day)
}

# Calculate alcohol intake for each participant
complete_meta$alcohol_g_day <- mapply(
  calculate_alcohol_g_day,
  complete_meta$drink_kind, complete_meta$drink_amount,
  complete_meta$drink_kind2, complete_meta$drink_amount2,
  complete_meta$alcohol_consumption
)


#### cigarettes


In [ ]:
complete_meta <- complete_meta %>%
  mutate(cigarettes_per_day = ifelse(smoking == "no", 0, cigarettes_per_day))


## add Metadata: serum interleukins

In [ ]:
# --- Serum cytokines ---
serum <- read_csv(str_glue("{path_meta}Serumproben_umformatiert.csv"))
serum <- serum %>% dplyr::select(-Status, -Anonym, -Well)
serum <- serum %>%
  rename(ID = Anonymisierung) %>%
  mutate(ID = str_pad(as.character(ID), width = 3, side = "left", pad = "0"))
complete_meta <- left_join(complete_meta, serum, by = "ID")

## add Metadata: glia markers

In [ ]:
# --- Glia markers ---
glia <- read_csv2(str_glue("{path_meta}Microglia_Darm-Hirn.csv"))
glia <- glia %>% dplyr::select(patho_number, IbA1, TMEM119, P2Y12)# keep relevant data
complete_meta <- left_join(complete_meta, glia, by = "patho_number")

# Glial markers: proportion of DAPI-positive nuclei -> percent
complete_meta <- complete_meta %>%
  mutate(across(c(IbA1, TMEM119, P2Y12), ~ .x * 100))

## Anonymize dataset

Delete all dates, some other identifying variables and the patho_number

In [ ]:
# Create anonymized dataset by removing original date variables and "patho_number"

anonym_meta <- subset(complete_meta, select = -c(birth_date, patho_number, date_of_admission, end_date, 
                                                 date_of_diagnosis, date_of_surgery, 
                                                 start_of_radiation, chemotherapy_start, 
                                                 date_of_relapse, blood_date, serum_date, stool_date, ID, 
                                                 food_allergies_kind, diet_kind_detail, drugs, age_at_diagnosis,
                                                 localisation, chemotherapy_drug, chemotherapy_dosage,
                                                 chemotherapy_cycle_numbers, intestinal_disease_which, radiation_dosage ))

anonym_meta$age<-cut(anonym_meta$age, 
                 breaks = seq(0, 100, by = 10), 
                 right = FALSE, # [0,10), [10,20) etc.
                 labels = paste(seq(0, 90, 10), seq(9, 99, 10), sep = "-"))

anonym_meta <- anonym_meta %>%
  arrange(age)


# converts back factor encoding to nummerical encoding (easier for import)
# -1 because original encodings started with 0 and R starts with 1
anonym_meta_conv <- data.frame(lapply(anonym_meta, function(x) {
  if (is.factor(x)) (as.numeric(x)-1) else x
}))

# Save anonymized dataset
write.csv(anonym_meta_conv, str_glue("{path_meta}MetadataGBM_v4.csv"), row.names = FALSE)

# Descriptive statistics

#### Comparison between groups

In [ ]:
descriptive_vars <- c("status","sex", "age", 
                    "cigarettes_per_day", "alcohol_g_day", "meat_consumption", "carbohydrates_consumption", "plants_consumption", "probiotics_frequency",
                    "CREA", "GFR", "GOT", "GPT", "CRP", "blood_glucose", "diabetes",
                    "antihypertensives", "anticoagulants", "gastric_protection", "analgesics", 
                    "antidiabetics", "sedatives", "statins", "prostate", "hormonal", 
                    "antiepileptics", "corticosteroids", "thyroid", "antidepressants",
                    "IFN_alpha", "IL_10", "IL_17AF", "IL_2", "IL_4", "IL_6", "TNF_alpha")


#### interleukins

In [ ]:
# Interleukin violin plots (patient vs control) with Wilcoxon p-values
interleukin_vars <- c("IFN_alpha", "IL_10", "IL_17AF", "IL_2", "IL_4", "IL_6", "TNF_alpha")

plot_list_il <- list()
for (var in interleukin_vars) {
  x <- complete_meta[[var]]
  g <- complete_meta$status
  ok <- !is.na(x) & !is.na(g)
  wt <- wilcox.test(x[ok] ~ g[ok], exact = FALSE)
  p_label <- ifelse(wt$p.value < 0.001, "p < 0.001",
                    paste0("p = ", formatC(wt$p.value, format = "f", digits = 2)))

  plot_list_il[[var]] <- ggplot(complete_meta[ok, ],
                                aes(x = status, y = .data[[var]], fill = status)) +
    geom_violin() +
    scale_fill_manual(values = col_pvc_raw) +
    scale_x_discrete(labels = pvc_labels) +
    labs(title = il_labels[[var]], y = "pg/ml", x = NULL, fill = NULL) +
    annotate("text", x = 1.5, y = max(x[ok], na.rm = TRUE),
             label = p_label, size = 2, vjust = 1) +
    theme_pub(8) +
    theme(plot.title = element_text(size = 6, face = "bold"),
          axis.title = element_text(size = 6),
          axis.text = element_text(size = 6),
          legend.position = "none")
}

combined_il <- wrap_plots(plot_list_il, ncol = 2, guides = "collect") &
    theme(legend.position = "bottom",
        legend.key.size = unit(3, "mm"),
        legend.text = element_text(size = 6),
        legend.margin = margin(0, 0, 0, 0),
        legend.box.margin = margin(-5, 0, 0, 0)) &
  scale_fill_manual(values = col_pvc_raw,
                                    labels = pvc_labels)

combined_il

save_fig(
  filename = "./output_glioblastoma/violin_interleukins_patient_vs_control.tiff",
  plot = combined_il,
  width = 8.5, height = 8.5, units = "cm", dpi = 300,
  compression = "lzw", bg = "white"
)

#### Descriptive data table 

In [ ]:
sel_meta_all<-complete_meta%>%dplyr::select(all_of(descriptive_vars))

In [ ]:
summary(complete_meta$status)

In [ ]:

# nummeric variables
num_vars <- sel_meta_all %>%
  dplyr::select(where(is.numeric)) %>%
  names()

# factor variables
fac_vars <- sel_meta_all %>%
  dplyr::select(-any_of(num_vars)) %>%
  dplyr::select(where(~ is.factor(.))) %>%
  names()


num_vars
fac_vars

In [ ]:
# Helper function: beautify variable names
beautify_varname <- function(varname) {
  varname %>%
    str_replace_all("_", " ") %>%
        str_replace_all("\\b([a-z])", toupper) %>%  # capitalize first letters, preserve existing capitals
    str_replace("^IbA1$", "Iba1")            # replace _ with space
}

# Numerical summary (mean ± sd)
num_summary <- function(var, group) {
  df <- data.frame(var = var, group = group)
  df_nonmiss <- df %>% filter(!is.na(var))

  grouped <- df_nonmiss %>%
    group_by(group) %>%
    summarise(med = median(var), q1 = quantile(var, 0.25), q3 = quantile(var, 0.75),
              .groups = "drop") %>%
    mutate(summary = sprintf("%.2f [%.2f–%.2f]", med, q1, q3)) %>%
    dplyr::select(group, summary) %>%
    pivot_wider(names_from = group, values_from = summary)

  total <- df_nonmiss %>%
    summarise(med = median(var), q1 = quantile(var, 0.25), q3 = quantile(var, 0.75)) %>%
    mutate(Total = sprintf("%.2f [%.2f–%.2f]", med, q1, q3)) %>%
    dplyr::select(Total)

  n_total <- nrow(df)
  n_miss <- sum(is.na(df$var))
  p_miss <- 100 * n_miss / n_total
  missing <- sprintf("%d (%.1f%%)", n_miss, p_miss)

  bind_cols(grouped, total, tibble(Missing = missing))
}

# Categorical summary (percentages)
cat_summary <- function(var, group, varname = NULL) {
  df <- data.frame(value = var, group = group)

  if (is.null(varname)) varname <- "Variable"

  df_nonmiss <- df %>% filter(!is.na(value))

  is_binary_yes_no <- length(unique(na.omit(var))) == 2 &&
    all(tolower(levels(factor(var))) %in% c("yes", "no"))

  grouped <- df_nonmiss %>%
    group_by(group, value) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(group) %>%
    mutate(p = n / sum(n)) %>%
    ungroup() %>%
    mutate(summary = sprintf("%d (%.1f%%)", n, p * 100))

  if (is_binary_yes_no) {
    grouped <- grouped %>% filter(tolower(value) == "yes")
  }

  grouped_wide <- grouped %>%
    dplyr::select(value, group, summary) %>%
    pivot_wider(names_from = group, values_from = summary, values_fill = "-")

  total <- df_nonmiss %>%
    group_by(value) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(p = n / sum(n),
           Total = sprintf("%d (%.1f%%)", n, p * 100)) %>%
    dplyr::select(value, Total)

  if (is_binary_yes_no) {
    total <- total %>% filter(tolower(value) == "yes")
    total$Variable <- varname
  } else {
    total$Variable <- paste0(varname, ": ", total$value)
  }

  n_total <- nrow(df)
  n_miss <- sum(is.na(df$value))
  p_miss <- 100 * n_miss / n_total
  total$Missing <- sprintf("%d (%.1f%%)", n_miss, p_miss)

  result <- left_join(total, grouped_wide, by = "value") %>%
    dplyr::select(Variable, everything(), -value)

  return(result)
}



# Define variable order and type for descriptive table
var_spec <- tribble(
  ~var,                          ~type,
  # Demographics
  "age",                         "num",
  "sex",                         "cat",
  # Lifestyle / Diet
  "cigarettes_per_day",          "num",
  "alcohol_g_day",               "num",
  "meat_consumption",            "cat",
  "carbohydrates_consumption",   "cat",
  "plants_consumption",          "cat",
  "probiotics_frequency",        "cat",
  # Metabolic comorbidities
  "diabetes",                    "cat",
  "blood_glucose",               "num",
  # Clinical chemistry
  "CREA",                        "num",
  "GFR",                         "num",
  "GOT",                         "num",
  "GPT",                         "num",
  "CRP",                         "num",
  # General medication
  "antihypertensives",           "cat",
  "anticoagulants",              "cat",
  "statins",                     "cat",
  "antidiabetics",               "cat",
  "gastric_protection",          "cat",
  "analgesics",                  "cat",
  "antidepressants",             "cat",
  "sedatives",                   "cat",
  "thyroid",                     "cat",
  "hormonal",                    "cat",
  "prostate",                    "cat",
  # Tumor-associated medication
  "corticosteroids",             "cat",
  "antiepileptics",              "cat"
)


# Create descriptive table in exact specified order
create_descriptive_table <- function(df, group_var, var_spec) {
  group_var <- enquo(group_var)
  group_vec <- df %>% pull(!!group_var)

  map_dfr(seq_len(nrow(var_spec)), function(i) {
    v    <- var_spec$var[i]
    type <- var_spec$type[i]
    if (type == "num") {
      res <- num_summary(df[[v]], group_vec)
      tibble(Variable = beautify_varname(v)) %>% bind_cols(res)
    } else {
      cat_summary(df[[v]], group_vec, varname = beautify_varname(v))
    }
  }) %>%
    mutate(across(everything(), ~ replace_na(as.character(.), "-")))
}


In [ ]:
summary(complete_meta)

In [ ]:
# Generate descriptive statistic table
table_out <- create_descriptive_table(
  df = complete_meta, group_var = status,
  var_spec = var_spec
)
table_out
write.csv(table_out[,1:4], file = "./output_glioblastoma/descriptive_summary_table_comparison_patient_controls.csv", row.names=F)

#### Patient characteristics

In [ ]:
patients_vars<- c("days_from_diagnosis_to_date_of_admission", "days_from_diagnosis_to_date_of_surgery", 
                    "karnofsky", "resection", "ki67", "methylation", "methylation_percentage", 
                    "radiation", "chemotherapy", "IbA1", "TMEM119", "P2Y12", "days_from_diagnosis_to_end_date")

In [ ]:
sel_meta_patients <- complete_meta %>%
  filter(status == "patient") %>%                  
  dplyr::select(all_of(patients_vars))      

In [ ]:
summary(sel_meta_patients)

<!-- ### Patient characteristics -->

In [ ]:
complete_meta_patients <- subset(complete_meta, status == "patient")
km_estimate<-survfit((Surv(days_from_diagnosis_to_end_date, event_status_death))~1,data=complete_meta_patients)
median_OS<-summary(km_estimate)$table["median"]
median_OS

In [ ]:

# nummeric variables
num_vars_p <- sel_meta_patients %>%
  dplyr::select(where(is.numeric)) %>%
  names()

# factor variables
fac_vars_p <- sel_meta_patients %>%
  dplyr::select(-any_of(num_vars_p)) %>%
  dplyr::select(where(~ is.factor(.))) %>%
  names()


num_vars_p
fac_vars_p

In [ ]:
# Create OS group variable

sel_meta_patients$OS_group <- ifelse(
  sel_meta_patients$days_from_diagnosis_to_end_date >= median_OS, "≥ median OS", "< median OS"
)

# Numerical summary (median [IQR]) stratified by group
num_summary_p_grouped <- function(var, group) {
  df <- data.frame(var = var, group = group)
  df_nonmiss <- df %>% filter(!is.na(var))
  grouped <- df_nonmiss %>%
    group_by(group) %>%
    summarise(med = median(var),
              q1 = unname(quantile(var, 0.25)),
              q3 = unname(quantile(var, 0.75)),
              .groups = "drop") %>%
    mutate(summary = sprintf("%.2f [%.2f–%.2f]", med, q1, q3)) %>%
    dplyr::select(group, summary) %>%
    pivot_wider(names_from = group, values_from = summary)
  total <- df_nonmiss %>%
    summarise(med = median(var),
              q1 = unname(quantile(var, 0.25)),
              q3 = unname(quantile(var, 0.75))) %>%
    mutate(Total = sprintf("%.2f [%.2f–%.2f]", med, q1, q3)) %>%
    dplyr::select(Total)
  n_total <- nrow(df)
  n_miss <- sum(is.na(df$var))
  p_miss <- 100 * n_miss / n_total
  missing <- sprintf("%d (%.1f%%)", n_miss, p_miss)
  bind_cols(grouped, total, tibble(Missing = missing))
}

# Categorical summary (percentages) stratified by group

cat_summary_p_grouped <- function(var, group, varname = NULL) {
  df <- data.frame(value = var, group = group)
  if (is.null(varname)) varname <- "Variable"
  varname <- beautify_varname(varname)
  df_nonmiss <- df %>% filter(!is.na(value))
  is_binary_yes_no <- length(unique(na.omit(var))) == 2 &&
    all(tolower(levels(factor(var))) %in% c("yes", "no"))
  grouped <- df_nonmiss %>%
    group_by(group, value) %>%
    summarise(n = n(), .groups = "drop") %>%
    group_by(group) %>%
    mutate(p = n / sum(n)) %>%
    ungroup() %>%
    mutate(summary = sprintf("%d (%.1f%%)", n, p * 100))
  if (is_binary_yes_no) {
    grouped <- grouped %>% filter(tolower(value) == "yes")
  }
  grouped_wide <- grouped %>%
    dplyr::select(value, group, summary) %>%
    pivot_wider(names_from = group, values_from = summary, values_fill = "-")
  total <- df_nonmiss %>%
    group_by(value) %>%
    summarise(n = n(), .groups = "drop") %>%
    mutate(p = n / sum(n),
           Total = sprintf("%d (%.1f%%)", n, p * 100))
  if (is_binary_yes_no) {
    total <- total %>% filter(tolower(value) == "yes")
    total$Variable <- varname
  } else {
    total$Variable <- paste0(varname, ": ", total$value)
  }
  total <- total %>% dplyr::select(value, Variable, Total)
  n_total <- nrow(data.frame(value = var))
  n_miss <- sum(is.na(var))
  p_miss <- 100 * n_miss / n_total
  total$Missing <- sprintf("%d (%.1f%%)", n_miss, p_miss)
  result <- left_join(total, grouped_wide, by = "value") %>%
    dplyr::select(Variable, everything(), -value)
  return(result)
}

 # Variable order for patient-only table
var_spec_p <- tribble(
  ~var,                                       ~type,
  # Clinical course & functional status
  "days_from_diagnosis_to_date_of_admission", "num",
  "days_from_diagnosis_to_date_of_surgery",   "num",
  "days_from_diagnosis_to_end_date",          "num",
  "karnofsky",                                "num",
  # Treatment
  "resection",                                "cat",
  "radiation",                                "cat",
  "chemotherapy",                             "cat",
  # Tumor biology
  "ki67",                                     "num",
  "methylation_percentage",                   "num",
  "methylation",                              "cat",
  # Glia markers (tumor microenvironment)
  "IbA1",                                     "num",
  "TMEM119",                                  "num",
  "P2Y12",                                    "num"
)

# Create descriptive table in exact specified order (patient only, grouped by OS)
create_descriptive_table_total_only_p <- function(df, var_spec, group_vec) {
  map_dfr(seq_len(nrow(var_spec)), function(i) {
    v    <- var_spec$var[i]
    type <- var_spec$type[i]
    if (type == "num") {
      res <- num_summary_p_grouped(df[[v]], group_vec)
      tibble(Variable = beautify_varname(v)) %>% bind_cols(res)
    } else {
      cat_summary_p_grouped(df[[v]], group_vec, varname = v)
    }
  }) %>%
    mutate(across(everything(), ~ replace_na(as.character(.), "-")))
}

In [ ]:
(sel_meta_patients$OS_group)


In [ ]:
# Generate descriptive statistic table
table_out <- create_descriptive_table_total_only_p(
  df        = sel_meta_patients,
  var_spec  = var_spec_p,
  group_vec = sel_meta_patients$OS_group
)

table_out
write.csv(table_out[,1:4], file = "./output_glioblastoma/descriptive_summary_table_patient_characteristics.csv", row.names=F)

In [ ]:
# Iba1/TMEM119/P2Y12 on percentage scale
sapply(complete_meta_patients[, c("IbA1","TMEM119","P2Y12")],
       function(x) round(quantile(x * 100, c(.25,.5,.75), na.rm = TRUE), 1))


# Survival analysis

In [ ]:
# generate table with only patients
complete_meta_patients <- subset(complete_meta, status == "patient")

## Kaplan-Meier



In [ ]:
km_estimate<-survfit((Surv(days_from_diagnosis_to_end_date, event_status_death))~1,data=complete_meta_patients)
km_estimate
summary(km_estimate)
median_OS<-summary(km_estimate)$table["median"]

In [ ]:
ggsurvplot(km_estimate, data = complete_meta_patients, risk.table = FALSE)

In [ ]:
# average amount of days per month times 6 month
365.25/2


In [ ]:
p_km_all<-ggsurvplot(
  km_estimate,
  data = complete_meta_patients,
  xscale = 30.44,
  break.time.by = 182.625,    
  xlab = "Time (months)",
  ylab = "Survival probability",
  surv.median.line = "hv",
  risk.table = FALSE,
  palette = "black",
  legend = "none"
)
p_km_all


save_fig(filename = "./output_glioblastoma/Kaplan_Meier_with_conf.tiff",
         plot = p_km_all$plot, width = W_SINGLE, height = 8, units = "cm",
         dpi = 300, compression = "lzw")

# Cox-regression

## Variables for Cox-regression

In [ ]:
# variables for regression analyses

# Rescale continuous variables  for better interpretability
  complete_meta_patients <- complete_meta_patients %>%
  mutate(
    age_at_diagnosis_10 = age_at_diagnosis / 10,
    karnofsky_10        = karnofsky / 10
  )

variables_cox <- c("age_at_diagnosis_10", "karnofsky_10", "resection",
                   "IbA1", "TMEM119", "P2Y12", "ki67", "methylation_percentage")

## Univariate variable selection

In [ ]:
# --- Univariate screening of candidate variables (console output + supplementary table) ---

a_priori_vars  <- c("age_at_diagnosis_10", "karnofsky_10", "resection")
screening_vars <- setdiff(variables_cox, a_priori_vars)

complete_meta_patients$resection <- relevel(complete_meta_patients$resection, ref = "total")

screen_one <- function(var) {

  fml <- as.formula(
    paste("Surv(days_from_diagnosis_to_end_date, event_status_death) ~", var)
  )

  fit <- tryCatch(
    withCallingHandlers(
      coxph(fml, data = complete_meta_patients),
      warning = function(w) {
        if (grepl("coefficient may be infinite|did not converge", w$message))
          invokeRestart("muffleWarning")
      }
    ),
    warning = function(w) NULL,
    error   = function(e) NULL
  )

  # Non-convergence or quasi-separation: report, do not select
  if (is.null(fit) || any(abs(coef(fit)) > 50, na.rm = TRUE)) {
    cat("Variable", var, "not estimable (non-convergence or quasi-separation).\n")
    return(tibble(Variable = var, n = NA_integer_, Events = NA_integer_,
                  `HR (95% CI)` = "not estimable",
                  `LRT p` = NA_real_, `Delta AIC` = NA_real_, Selected = "no"))
  }

  # Null model on the same complete cases
  dat <- complete_meta_patients[complete.cases(
    complete_meta_patients[, c("days_from_diagnosis_to_end_date",
                               "event_status_death", var)]), ]
  fit_null <- coxph(Surv(days_from_diagnosis_to_end_date, event_status_death) ~ 1,
                    data = dat)

  lrt_p <- anova(fit_null, fit, test = "LRT")[2, "Pr(>|Chi|)"]
  d_aic <- AIC(fit)-AIC(fit_null)
  sel   <- !is.na(lrt_p) && lrt_p < 0.05 && d_aic <= -2

  cat("Variable", var, if (sel) "selected." else "not selected.",
      "LRT p =", round(lrt_p, 4), "| delta AIC =", round(d_aic, 2), "\n")
  if (sel) print(summary(fit))

  s  <- summary(fit)
  ci <- s$conf.int

  # One row per coefficient - relevant for multi-level factors
  purrr::map_dfr(seq_len(nrow(ci)), function(i) {
    tibble(
      Variable      = if (nrow(ci) == 1) var else paste0(var, ": ", rownames(ci)[i]),
      n             = fit$n,
      Events        = fit$nevent,
      `HR (95% CI)` = sprintf("%.2f (%.2f\u2013%.2f)",
                              ci[i, "exp(coef)"], ci[i, "lower .95"], ci[i, "upper .95"]),
      `LRT p`       = if (i == 1) round(lrt_p, 4) else NA_real_,
      `Delta AIC`   = if (i == 1) round(d_aic, 2) else NA_real_,
      Selected      = if (i == 1) ifelse(sel, "yes", "no") else NA_character_
    )
  })
}

screening_tab <- purrr::map_dfr(screening_vars, screen_one)

selected_vars <- screening_tab %>%
  dplyr::filter(Selected == "yes") %>%
  dplyr::pull(Variable)

final_vars <- c(a_priori_vars, selected_vars)
cat("\nA priori variables:", paste(a_priori_vars, collapse = ", "), "\n")
cat("Data-driven variables:", paste(selected_vars, collapse = ", "), "\n")
cat("Final model variables:", paste(final_vars, collapse = ", "), "\n")

screening_tab
write.csv(screening_tab, "./output_glioblastoma/TableS_cox_screening.csv", row.names = FALSE)

### Check pre-selected variables 

age

In [ ]:
summary(complete_meta_patients$age_at_diagnosis_10)

In [ ]:
#age
cox_age<-coxph(Surv(days_from_diagnosis_to_end_date, event_status_death) ~age_at_diagnosis_10, data=complete_meta_patients)
rr_cox_age<-resid(cox_age)

meta_age <- filter(complete_meta_patients, !is.na(age))

plot(meta_age$age, rr_cox_age, xlab= "age ", ylab="residuals")


Karnofsky

In [ ]:
summary(complete_meta_patients$karnofsky_10)

In [ ]:
#karnofsky_10
cox_karnofsky_10<-coxph(Surv(days_from_diagnosis_to_end_date, event_status_death) ~karnofsky_10, data=complete_meta_patients)
rr_cox_karnofsky_10<-resid(cox_karnofsky_10)

meta_karnofsky_10 <- filter(complete_meta_patients, !is.na(karnofsky_10))

plot(meta_karnofsky_10$karnofsky_10, rr_cox_karnofsky_10, xlab= "karnofsky_10 ", ylab="residuals")


resection

In [ ]:
summary(complete_meta_patients$resection)

In [ ]:
#resection
cox_resection<-coxph(Surv(days_from_diagnosis_to_end_date, event_status_death) ~resection, data=complete_meta_patients)
rr_cox_resection<-resid(cox_resection)

meta_resection <- filter(complete_meta_patients, !is.na(resection))

plot(meta_resection$resection, rr_cox_resection, xlab= "Resection ", ylab="residuals")


Iba1

In [ ]:
summary(complete_meta_patients$IbA1)

In [ ]:
cox_IbA1<-coxph(Surv(days_from_diagnosis_to_end_date, event_status_death) ~IbA1, data=complete_meta_patients)
rr_cox_IbA1<-resid(cox_IbA1)

meta_IbA1 <- filter(complete_meta_patients, !is.na(IbA1))

plot(meta_IbA1$IbA1, rr_cox_IbA1, xlab= "IbA1 (per 1 % of cells)", ylab="Residuals")
lines(lowess(meta_IbA1$IbA1, rr_cox_IbA1,iter=0),lty=2)

### Primary Cox model

In [ ]:
# Multivariable Cox model with univariate-selected variables + Iba1
cox_multi <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + resection + age_at_diagnosis_10 + karnofsky_10,
  data = complete_meta_patients
)

summary(cox_multi)



Iba1 significantly associated with outcome after correction for known risk factors age, karnofsky and resection

### Check model assumptions

In [ ]:
# Check proportional hazards assumption
cox_zph <- cox.zph(cox_multi)
print(cox_zph)
plot(cox_zph)

In [ ]:
# Martingale residuals from null model vs. each continuous predictor
cox_null <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~ 1,
  data = complete_meta_patients %>% filter(!is.na(IbA1))
)
res_null <- resid(cox_null, type = "martingale")

par(mfrow = c(1, 4))
# Iba1
plot(complete_meta_patients %>% filter(!is.na(IbA1)) %>% pull(IbA1),
     res_null, xlab = "IbA1 (per 1 % of cells)", ylab = "Martingale residuals")
lines(lowess(complete_meta_patients %>% filter(!is.na(IbA1)) %>% pull(IbA1), res_null), lty = 2)

# Age
plot(complete_meta_patients %>% filter(!is.na(IbA1)) %>% pull(age_at_diagnosis_10),
     res_null, xlab = "Age (per 10 y)", ylab = "Martingale residuals")
lines(lowess(complete_meta_patients %>% filter(!is.na(IbA1)) %>% pull(age_at_diagnosis_10), res_null), lty = 2)

# Karnofsky
plot(complete_meta_patients %>% filter(!is.na(IbA1)) %>% pull(karnofsky_10),
     res_null, xlab = "Karnofsky (per 10 pts)", ylab = "Martingale residuals")
lines(lowess(complete_meta_patients %>% filter(!is.na(IbA1)) %>% pull(karnofsky_10), res_null), lty = 2)

# Resection
plot(complete_meta_patients %>% filter(!is.na(IbA1)) %>% pull(resection),
     res_null, xlab = "Resection", ylab = "Martingale residuals")

In [ ]:
# dfbeta residuals for primary model
dfb <- resid(cox_multi, type = "dfbeta")
par(mfrow = c(2, 3))
for (j in 1:ncol(dfb)) {
  plot(dfb[, j], ylab = paste("dfbeta:", colnames(dfb)[j]),
       xlab = "Observation", pch = 19)
  abline(h = 0, lty = 2)
}

In [ ]:
# Identify most influential observations for IbA1
dfb <- resid(cox_multi, type = "dfbeta")
colnames(dfb) <- names(coef(cox_multi))
which(abs(dfb[, "IbA1"]) > 0.3)
dfb

no single observations with high influence for IbA1

In [ ]:
# Leave-one-out sensitivity across all observations used in the primary model
dat_multi <- model.frame(cox_multi)
loo_tab <- purrr::map_dfr(seq_len(nrow(dat_multi)), function(i) {
  fit_i <- coxph(
    Surv(days_from_diagnosis_to_end_date, event_status_death) ~
      IbA1 + resection + age_at_diagnosis_10 + karnofsky_10,
    data = complete_meta_patients[complete.cases(
      complete_meta_patients[, c("days_from_diagnosis_to_end_date", "event_status_death",
                                 "IbA1", "resection",
                                 "age_at_diagnosis_10", "karnofsky_10")]), ][-i, ]
  )
  s <- summary(fit_i)
  tibble(omitted = i,
         HR      = s$conf.int["IbA1", "exp(coef)"],
         lower   = s$conf.int["IbA1", "lower .95"],
         upper   = s$conf.int["IbA1", "upper .95"],
         p       = s$coefficients["IbA1", "Pr(>|z|)"])
})

range(loo_tab$HR); range(loo_tab$p)
loo_tab

HR for IbA1 is robust, p-value influenced due to reduction of observations

In [ ]:
# Variance inflation factors

vif(cox_multi)

### Sensitivity analysis: parsimonious models

In [ ]:

# Model 1: IbA1 + Age
cox_sens1 <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + age_at_diagnosis_10,
  data = complete_meta_patients
)
summary(cox_sens1)

# Model 2: IbA1 + Karnofsky
cox_sens2 <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + karnofsky_10,
  data = complete_meta_patients
)
summary(cox_sens2)


# Model 3: IbA1 + Resection
cox_sens3 <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + resection,
  data = complete_meta_patients
)
summary(cox_sens3)

# Model 4: IbA1 + Age + Resection
cox_sens4 <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + age_at_diagnosis_10 + resection,
  data = complete_meta_patients
)
summary(cox_sens4)

# Model 5: IbA1 + Karnofsky + Resection
cox_sens5 <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + karnofsky_10 + resection,
  data = complete_meta_patients
)
summary(cox_sens5)


# Model 6: IbA1 + Karnofsky + Age
cox_sens6 <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + age_at_diagnosis_10 + karnofsky_10,
  data = complete_meta_patients
)
summary(cox_sens6)


In [ ]:
# Check PH assumption for all sensitivity models
cox.zph(cox_sens1)
cox.zph(cox_sens2)
cox.zph(cox_sens3)
cox.zph(cox_sens4)
cox.zph(cox_sens5)
cox.zph(cox_sens6)

### Sensitivity analysis for methylation

In [ ]:
# Model 1: IbA1 + Methylation
cox_sens7 <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + methylation,
  data = complete_meta_patients
)
summary(cox_sens7)

In [ ]:
# Model 1: primary model + Methylation
cox_sens8  <- coxph(
  Surv(days_from_diagnosis_to_end_date, event_status_death) ~
    IbA1 + resection + age_at_diagnosis_10 + karnofsky_10 + methylation,
  data = complete_meta_patients
)
summary(cox_sens8)

In [ ]:
# Check PH assumption for all sensitivity models
cox.zph(cox_sens7)
cox.zph(cox_sens8)

In [ ]:
# Extract HR (CI) string for a variable from a model, or "-" if not in model
extract_hr_ci <- function(model, varname) {
  s <- summary(model)
  coefs <- as.data.frame(s$coefficients)
  ci <- as.data.frame(s$conf.int)
  if (varname %in% rownames(coefs)) {
    sprintf("%.2f (%.3f–%.3f)",
            coefs[varname, "exp(coef)"],
            ci[varname, "lower .95"],
            ci[varname, "upper .95"])
  } else {
    "-"
  }
}

# Extract p-value for a variable
extract_p <- function(model, varname) {
  s <- summary(model)
  coefs <- as.data.frame(s$coefficients)
  if (varname %in% rownames(coefs)) {
    pval <- coefs[varname, "Pr(>|z|)"]
    if (pval < 0.001) return("<0.001")
    sprintf("%.3f", pval)
  } else {
    "-"
  }
}

# Build table row with IbA1 estimates 
build_row <- function(model, model_name) {
  tibble(
    Model = model_name,
    `Iba1 (per 1%) HR (95% CI)` = extract_hr_ci(model, "IbA1"),
    `Iba1 p-value` = extract_p(model, "IbA1")
  )
}

cox_summary_table <- bind_rows(
  build_row(cox_multi,  "Primary (Iba1 + Resection + Age + Karnofsky)"),
  build_row(cox_sens1,  "Iba1 + Age"),
  build_row(cox_sens2,  "Iba1 + Karnofsky"),
  build_row(cox_sens3,  "Iba1 + Resection"),
  build_row(cox_sens4,  "Iba1 + Age + Resection"),
  build_row(cox_sens5,  "Iba1 + Karnofsky + Resection"),
  build_row(cox_sens6,  "Iba1 + Age + Karnofsky"),
  build_row(cox_sens7,  "Iba1 + Methylation"),
  build_row(cox_sens8,  "Iba1 + Age + Karnofsky + Resection + Methylation")
)

print(cox_summary_table)

write.csv(cox_summary_table,
          file = "./output_glioblastoma/cox_sensitivity_table.csv",
          row.names = FALSE)

### Visualisation primary Cox model

In [ ]:
# Font sizes (adjust to match your publication)
txt_mm <- 2.8
txt_pt <- 8

# Extract coefficients
s <- summary(cox_multi)
coefs <- as.data.frame(s$coefficients)
ci <- as.data.frame(s$conf.int)

df_cox_forest <- tibble(
  term    = rownames(coefs),
  HR      = coefs[, "exp(coef)"],
  CI_low  = ci[, "lower .95"],
  CI_high = ci[, "upper .95"],
  p_value = coefs[, "Pr(>|z|)"]
)

# Label mapping
label_map_cox <- c(
  "IbA1"             = "Iba1 (per 1%)",
  "resectionbiopsy"     = "Resection: biopsy (ref total)",
  "resectionsubtotal"   = "Resection: subtotal (ref total)",
  "age_at_diagnosis_10" = "Age at diagnosis (per 10 y)",
  "karnofsky_10"        = "Karnofsky (per 10 pts)"
)

df_cox_forest <- df_cox_forest %>%
  mutate(
    label     = recode(term, !!!label_map_cox, .default = term),
    y_num     = row_number(),
    or_ci_txt = sprintf("%.2f (%.2f\u2013%.2f)", HR, CI_low, CI_high),
    p_txt     = case_when(
      p_value < 0.001 ~ "<0.001",
      p_value < 0.01  ~ sprintf("%.3f", p_value),
      TRUE            ~ sprintf("%.2f",  p_value)
    )
  )

y_max   <- max(df_cox_forest$y_num)
x_limits <- c(0.01, 100)

# Header
p_header <- ggplot() +
  annotate("text", x = 0,    y = 0, label = "Variable",
           hjust = 0, fontface = "bold", size = txt_mm) +
  annotate("text", x = 0.68, y = 0, label = "Hazard ratio",
           hjust = 0, fontface = "bold", size = txt_mm) +
  annotate("text", x = 1.08, y = 0, label = "HR (95% CI)",
           hjust = 0, fontface = "bold", size = txt_mm) +
  annotate("text", x = 1.28, y = 0, label = "p-value",
           hjust = 0, fontface = "bold", size = txt_mm) +
  scale_x_continuous(limits = c(0, 1.40),
                     expand = expansion(mult = c(0, 0))) +
  scale_y_continuous(limits = c(-1, 1),
                     expand = expansion(mult = c(0, 0))) +
  theme_void() +
  theme(plot.margin = margin(t = 6, r = 6, b = 6, l = 6))

# Panel A: labels
p_labels <- ggplot(df_cox_forest, aes(y = y_num)) +
  geom_text(aes(x = 0, label = label), hjust = 0,
            size = txt_mm, fontface = "bold") +
  scale_x_continuous(limits = c(0, 1),
                     expand = expansion(mult = c(0, 0))) +
  scale_y_reverse(limits = c(y_max + 0.6, 0.4)) +
  theme_void() +
  theme(plot.margin = margin(t = 0, r = 6, b = 6, l = 6))

# Panel B: forest graph with frame and grid lines
p_forest <- ggplot(df_cox_forest, aes(x = HR, y = y_num)) +
  geom_vline(xintercept = 1, linetype = 2, linewidth = 0.4) +
  geom_errorbarh(aes(xmin = CI_low, xmax = CI_high),
                 height = 0.18, linewidth = 0.5) +
  geom_point(size = 2.6) +
  scale_x_log10(limits = x_limits) +
  scale_y_reverse(limits = c(y_max + 0.6, 0.4)) +
  labs(x = NULL, y = NULL) +
  theme_bw() +
  theme(
    axis.title.y       = element_blank(),
    axis.text.y        = element_blank(),
    axis.ticks.y       = element_blank(),
    panel.grid.major.y = element_blank(),
    panel.grid.minor.y = element_blank(),
    axis.text.x        = element_text(size = txt_pt),
    plot.margin        = margin(t = 0, r = 6, b = 6, l = 6)
  )

# Panel C: stats text
p_stats <- ggplot(df_cox_forest, aes(y = y_num)) +
  geom_text(aes(x = 0, label = or_ci_txt), hjust = 0, size = txt_mm) +
  geom_text(aes(x = 1, label = p_txt),     hjust = 0, size = txt_mm) +
  scale_x_continuous(limits = c(0, 1.40),
                     expand = expansion(mult = c(0, 0))) +
  scale_y_reverse(limits = c(y_max + 0.6, 0.4)) +
  theme_void() +
  theme(plot.margin = margin(t = 0, r = 6, b = 6, l = 6))

# Combine with patchwork
p_body <- p_labels + p_forest + p_stats +
  plot_layout(widths = c(0.35, 0.40, 0.25))

p_cox_forest_final <- p_header / p_body +
  plot_layout(heights = c(0.10, 0.90))

print(p_cox_forest_final)

save_fig(
  filename    = "./output_glioblastoma/Cox_multivariable_forest_plot.tiff",
  plot        = p_cox_forest_final,
  width       = 17.4,
  height      = 8,
  units       = "cm",
  dpi         = 300,
  compression = "lzw",
  bg          = "white"
)

## Prepare to add variables to phyloseq object 
Add survival/follow up time (days_from_diagnosis_to_end_date) and variable for OS_group (lower/higher than median) to sample_data of phyloseq object

In [ ]:
complete_meta_patients_OS <- complete_meta_patients
complete_meta_patients_OS$OS_group<-ifelse(complete_meta_patients$days_from_diagnosis_to_end_date >= median_OS, "higherOS", "lowerOS")
complete_meta_patients_OS$OS_group<-factor(complete_meta_patients_OS$OS_group, levels=c("lowerOS", "higherOS"))
names(complete_meta_patients_OS)[names(complete_meta_patients_OS) == "ID"] <- "sample"
head(complete_meta_patients_OS)

In [ ]:
meta_patients_OS<-complete_meta_patients_OS[c("sample", "OS_group","days_from_diagnosis_to_end_date")]
meta_patients_OS$OS<-(meta_patients_OS$days_from_diagnosis_to_end_date/30.44) #convert to month
meta_patients_OS<-meta_patients_OS[,-3]
meta_patients_OS

# Interleukins for patients by OS

In [ ]:
## Interleukin violin plots (lower OS vs higher OS, patients only) with Wilcoxon p-values
interleukin_vars <- c("IFN_alpha", "IL_10", "IL_17AF", "IL_2", "IL_4", "IL_6", "TNF_alpha")

# Color scheme for OS groups
os_colors <- col_os_raw


# Restrict to patients with non-missing OS_group
patients_os <- complete_meta_patients_OS %>%
  dplyr::filter(!is.na(OS_group))

plot_list_il_os <- list()
for (var in interleukin_vars) {
  x  <- patients_os[[var]]
  g  <- patients_os$OS_group
  ok <- !is.na(x) & !is.na(g)
  wt <- wilcox.test(x[ok] ~ g[ok], exact = FALSE)
  p_label <- ifelse(wt$p.value < 0.001, "p < 0.001",
                    paste0("p = ", formatC(wt$p.value, format = "f", digits = 2)))
  plot_list_il_os[[var]] <- ggplot(patients_os[ok, ],
                                   aes(x = OS_group, y = .data[[var]], fill = OS_group)) +
    geom_violin() +
    scale_fill_manual(values = os_colors) +
    scale_x_discrete(labels = os_labels) +
    labs(title = il_labels[[var]], y = "pg/ml", x = NULL, fill = NULL) +
    labs(title = il_labels[[var]], y = "pg/ml", x = NULL, fill = NULL) +
    annotate("text", x = 1.5, y = max(x[ok], na.rm = TRUE),
             label = p_label, size = 2, vjust = 1) +
    theme_pub(8)+
    theme(plot.title  = element_text(size = 6, face = "bold"),
          axis.title  = element_text(size = 6),
          axis.text   = element_text(size = 6),
          legend.position = "none")
}

combined_il_os <- wrap_plots(plot_list_il_os, ncol = 2, guides = "collect") &
  theme(legend.position    = "bottom",
        legend.key.size    = unit(3, "mm"),
        legend.text        = element_text(size = 6),
        legend.margin      = margin(0, 0, 0, 0),
        legend.box.margin  = margin(-5, 0, 0, 0)) &
  scale_fill_manual(values = os_colors, labels = os_labels)

combined_il_os

save_fig(
  filename     = "./output_glioblastoma/violin_interleukins_lowerOS_vs_higherOS.tiff",
  plot         = combined_il_os,
  width        = 8.5, height = 8.5, units = "cm", dpi = 300,
  compression  = "lzw", bg = "white"
)

# Analysis microbiome data

## Patients vs controls

### Diversity

In [ ]:
# Find taxa that appear in many (ideally all) samples
# Count in how many samples each taxon has at least one read
prevalence_ITS <- apply(otu_table(phy_ITS_spc_filtered), 1, function(x) sum(x > 0))

# Tax info for context
tax_ITS <- as.data.frame(tax_table(phy_ITS_spc_filtered))

# Top taxa by prevalence
n_samples_ITS <- nsamples(phy_ITS_spc_filtered)

# Show the most prevalent ITS taxa with their species names
data.frame(
  taxon_id = names(prevalence_ITS),
  n_samples_present = prevalence_ITS,
  pct_samples = round(100 * prevalence_ITS / n_samples_ITS, 1),
  total_reads = taxa_sums(phy_ITS_spc_filtered),
  species = tax_ITS$species
) %>%
  arrange(desc(n_samples_present), desc(total_reads)) %>%
  head(15)

In [ ]:
# ITS Patients vs Controls, use S. cerevisiae as base
set.seed(123)
divnet_ITS_pc <- divnet(phy_ITS_spc_filtered, X = "type", base = "ASV2", ncores=n_cores , tuning = "careful")


In [ ]:
# DivNet fits for Patients vs Controls
# - Genus level for 16S, species level for ITS 
# - Default settings
# - X = "type" uses the c/p grouping in sample_data

set.seed(123)
# 16S Patients vs Controls
divnet_16S_pc <- divnet(phy_16S_gen_filtered, X = "type", ncores=n_cores , tuning = "careful")

#### Alpha diversity

In [ ]:
# Helper: extract group-effect p-value from testDiversity() output
get_p_from_testdiv <- function(td_result) {
  as.numeric(td_result[2, "p-values"])
}

# Run testDiversity for ITS and 16S, store p-values
td_its_shannon <- testDiversity(divnet_ITS_pc, h0 = "shannon")
td_its_simpson <- testDiversity(divnet_ITS_pc, h0 = "simpson")
td_16s_shannon <- testDiversity(divnet_16S_pc, h0 = "shannon")
td_16s_simpson <- testDiversity(divnet_16S_pc, h0 = "simpson")


td_16s_shannon; td_16s_simpson; td_its_shannon; td_its_simpson

pvals_its <- c(
  Shannon          = get_p_from_testdiv(td_its_shannon),
  Simpson = get_p_from_testdiv(td_its_simpson)
)
pvals_16s <- c(
  Shannon          = get_p_from_testdiv(td_16s_shannon),
  Simpson = get_p_from_testdiv(td_16s_simpson)
)

# Helper: build one-row-per-group summary from DivNet output
summarise_alpha_per_group <- function(divnet_obj, ps, group_var) {
  meta   <- as(sample_data(ps), "data.frame")
  groups <- meta[[group_var]]
  names(groups) <- rownames(meta)
  un <- function(x) unname(as.numeric(x))

  build_index <- function(div_idx, index_name) {
    samples <- names(div_idx)
    df <- tibble(
      sample   = samples,
      group    = groups[samples],
      estimate = vapply(div_idx, function(x) un(x$estimate),    numeric(1)),
      lower    = vapply(div_idx, function(x) un(x$interval[1]), numeric(1)),
      upper    = vapply(div_idx, function(x) un(x$interval[2]), numeric(1))
    )
    df %>%
      group_by(group) %>%
      summarise(
        estimate = first(estimate),
        lower    = first(lower),
        upper    = first(upper),
        .groups  = "drop"
      ) %>%
      mutate(index = index_name)
  }

  bind_rows(
    build_index(divnet_obj$shannon, "Shannon"),
    build_index(divnet_obj$simpson, "Simpson")
  ) %>%
    dplyr::select(group, index, estimate, lower, upper)
}

# Plot: one point per group with 95% CI, faceted by index, with p-value annotation
plot_alpha_grouped <- function(alpha_df,
                               group_colors,
                               pvals       = NULL,
                               title       = NULL,
                               caption     = NULL,
                               x_label     = NULL,
                               y_label     = "Estimate",
                               base_size   = 8,
                               point_size  = 2,
                               errbar_w    = 0.15,
                               errbar_lw   = 0.5,
                               pval_size   = NULL) {
  if (is.null(pval_size)) pval_size <- base_size * 0.3

  p <- ggplot(alpha_df, aes(x = group, y = estimate, color = group)) +
    geom_point(size = point_size) +
    geom_errorbar(aes(ymin = lower, ymax = upper),
                  width = errbar_w, linewidth = errbar_lw) +
    facet_wrap(~ index, scales = "free_y") +
    scale_color_manual(values = group_colors) +
    labs(x = x_label, y = y_label, title = title, caption = caption) +
    theme_pub(8) +
    theme(legend.position  = "none",
          panel.grid.minor = element_blank(),
          plot.caption     = element_text(hjust = 0, size = base_size - 1,
                                          margin = margin(t = 4)),
          plot.title       = element_text(face = "bold"),
          strip.text       = element_text(face = "bold"))

  if (!is.null(pvals)) {
    fmt_p <- function(p) {
      if (p < 0.001) "italic(p) < 0.001"
      else sprintf("italic(p) == %.3f", p)
    }
    pval_df <- tibble(
      index = names(pvals),
      label = vapply(pvals, fmt_p, character(1))
    )
    p <- p + geom_text(
      data = pval_df,
      aes(x = -Inf, y = Inf, label = label),
      hjust = -1.15, vjust = 1,
      inherit.aes = FALSE,
      parse = TRUE,
      size  = pval_size
    )
  }
  p
}

# Subset phyloseq objects to patient/control samples (matching the divnet fits)
phy_ITS_pc <- subset_samples(phy_ITS_spc_filtered, type %in% c("c", "p"))
phy_16S_pc <- subset_samples(phy_16S_gen_filtered, type %in% c("c", "p"))

# Build summary tables and recode group labels
alpha_df_its <- summarise_alpha_per_group(
  divnet_obj = divnet_ITS_pc,
  ps         = phy_ITS_pc,
  group_var  = "type"
) %>%
  mutate(group = recode(group, c = "Controls", p = "Patients"),
         group = factor(group, levels = c("Controls", "Patients")))

alpha_df_16s <- summarise_alpha_per_group(
  divnet_obj = divnet_16S_pc,
  ps         = phy_16S_pc,
  group_var  = "type"
) %>%
  mutate(group = recode(group, c = "Controls", p = "Patients"),
         group = factor(group, levels = c("Controls", "Patients")))

# Group colors
group_colors_pc <- col_pvc   # central PvC palette

# Build the plots — defaults tuned for half-page (single column) journal figures
plot_its <- plot_alpha_grouped(
  alpha_df_its,
  group_colors = group_colors_pc,
  pvals        = pvals_its,
  title        = "ITS alpha diversity",
  caption      = NULL,
  base_size    = 8,
  point_size   = 2,
  errbar_w     = 0.12
)

plot_16s <- plot_alpha_grouped(
  alpha_df_16s,
  group_colors = group_colors_pc,
  pvals        = pvals_16s,
  title        = "16S alpha diversity",
  caption      = NULL,
  base_size    = 8,
  point_size   = 2,
  errbar_w     = 0.12
)

p_plot_16s <- plot_16s + scale_y_continuous(expand = expansion(mult = c(0.05, 0.20)))
p_plot_its <- plot_its + scale_y_continuous(expand = expansion(mult = c(0.05, 0.50)))

# Save at single-column journal width (8.5 cm wide, ~6 cm tall)
save_fig("./output_glioblastoma/alpha_ITS_pc.pdf",  plot_its, width = 8.5, height = 6, units = "cm")
save_fig("./output_glioblastoma/alpha_16S_pc.pdf",  plot_16s, width = 8.5, height = 6, units = "cm")
# TIFF version at 600 dpi with LZW compression (journal standard)
save_fig("./output_glioblastoma/alpha_ITS_pc.tiff", plot_its, width = 8.5, height = 6, units = "cm",
       dpi = 600, compression = "lzw")
save_fig("./output_glioblastoma/alpha_16S_pc.tiff", plot_16s, width = 8.5, height = 6, units = "cm",
       dpi = 600, compression = "lzw")

plot_its
plot_16s

#### Bray-Curtis

ITS

In [ ]:
#construct matrix
n <- nsamples(phy_ITS_pc)
X_diag <- diag(n)
colnames(X_diag) <- rownames(X_diag) <- as.character(1:n)

# calculate bray-curtis
set.seed(123)
divnet_ITS_pc_beta <- divnet(W = phy_ITS_pc,
                             X = X_diag,
                             variance = "none", base="ASV2", tuning="careful")

# test bray-curtis
set.seed(123)
bc_test_ITS_pc<-testBetaDiversity(divnet_ITS_pc_beta,
                  h0 = "bray-curtis",
                  n_boot = 10000,
                  sample_specimen_matrix = X_diag,
                  groups = sample_data(phy_ITS_pc)$type)

bc_test_ITS_pc$p_value                  


16S

In [ ]:
#construct matrix
n <- nsamples(phy_16S_pc)
X_diag <- diag(n)
colnames(X_diag) <- rownames(X_diag) <- as.character(1:n)

set.seed(123)
# calculate bray-curtis
divnet_16S_pc_beta <- divnet(W = phy_16S_pc,
                             X = X_diag,
                             variance = "none", tuning="careful")

# test bray-curtis
set.seed(123)
bc_test_16S_pc<-testBetaDiversity(divnet_16S_pc_beta,
                  h0 = "bray-curtis",
                  n_boot = 10000,
                  sample_specimen_matrix = X_diag,
                  groups = sample_data(phy_16S_pc)$type)

bc_test_16S_pc$p_value                  


In [ ]:
# Extract Bray-Curtis distance matrix from DivNet (ITS, patient vs control)
bc_mat_ITS_pc <- divnet_ITS_pc_beta[["bray-curtis"]]

# Numerical noise on diagonal: clamp small negatives/values to 0
bc_mat_ITS_pc[bc_mat_ITS_pc < 0] <- 0
diag(bc_mat_ITS_pc) <- 0
bc_dist_ITS_pc <- as.dist(bc_mat_ITS_pc)

# PCoA (Cailliez correction handles negative eigenvalues from non-Euclidean BC)
pcoa_res_ITS_pc <- ape::pcoa(bc_dist_ITS_pc, correction = "cailliez")

# Variance explained — use corrected eigenvalues if Cailliez was applied
eig_col_ITS_pc  <- if ("Rel_corr_eig" %in% colnames(pcoa_res_ITS_pc$values))
                     "Rel_corr_eig" else "Relative_eig"
var_expl_ITS_pc <- round(100 * pcoa_res_ITS_pc$values[[eig_col_ITS_pc]][1:2], 1)

# Build plotting data
meta_ITS_pc <- as(sample_data(phy_ITS_pc), "data.frame") %>%
  rownames_to_column("sample_id")

pcoa_df_ITS_pc <- as.data.frame(pcoa_res_ITS_pc$vectors[, 1:2]) %>%
  rownames_to_column("sample_id") %>%
  rename(Axis1 = Axis.1, Axis2 = Axis.2) %>%
  left_join(meta_ITS_pc, by = "sample_id") %>%
  mutate(group = recode(type, c = "Controls", p = "Patients"),
         group = factor(group, levels = c("Controls", "Patients")))

# p-value annotation
p_val_ITS_pc   <- bc_test_ITS_pc$p_value
p_label_ITS_pc <- if (p_val_ITS_pc < 0.001) "italic(p) < 0.001" else
                                            sprintf("italic(p) == %.3f", p_val_ITS_pc)

# Plot
plot_pcoa_ITS_pc <- ggplot(pcoa_df_ITS_pc,
                           aes(x = Axis1, y = Axis2,
                               color = group, fill = group)) +
  stat_ellipse(geom = "polygon", alpha = 0.15,
               linewidth = 0.4, level = 0.95) +
  geom_point(size = 2) +
  scale_color_manual(values = group_colors_pc, name = NULL) +
  scale_fill_manual(values  = group_colors_pc, name = NULL) +
  labs(x = sprintf("PCoA 1 (%.1f%%)", var_expl_ITS_pc[1]),
       y = sprintf("PCoA 2 (%.1f%%)", var_expl_ITS_pc[2]),
       color = NULL, fill = NULL,
       title   = NULL,
       caption = NULL) +
  annotate("text", x = -Inf, y = Inf, label = p_label_ITS_pc,
           hjust = 0, vjust = 1, parse = TRUE,
           size = 8 * 0.36) +
  theme_pub(8) +
  theme(legend.position  = "bottom",
        panel.grid.minor = element_blank(),
        plot.title       = element_text(face = "bold"),
        plot.caption     = element_text(hjust = 0, size = 7,
                                        margin = margin(t = 4)))

# Save (single-column journal width)
save_fig("./output_glioblastoma/pcoa_ITS_pc.pdf",  plot_pcoa_ITS_pc, width = 8.5, height = 8.5, units = "cm")
save_fig("./output_glioblastoma/pcoa_ITS_pc.tiff", plot_pcoa_ITS_pc, width = 8.5, height = 8.5, units = "cm",
       dpi = 600, compression = "lzw")

plot_pcoa_ITS_pc

In [ ]:
# Extract Bray-Curtis distance matrix from DivNet (16S, patient vs control)
bc_mat_16S_pc <- divnet_16S_pc_beta[["bray-curtis"]]

# Numerical noise on diagonal: clamp small negatives/values to 0
bc_mat_16S_pc[bc_mat_16S_pc < 0] <- 0
diag(bc_mat_16S_pc) <- 0
bc_dist_16S_pc <- as.dist(bc_mat_16S_pc)

# PCoA (Cailliez correction handles negative eigenvalues from non-Euclidean BC)
pcoa_res_16S_pc <- ape::pcoa(bc_dist_16S_pc, correction = "cailliez")

# Variance explained — use corrected eigenvalues if Cailliez was applied
eig_col_16S_pc  <- if ("Rel_corr_eig" %in% colnames(pcoa_res_16S_pc$values))
                     "Rel_corr_eig" else "Relative_eig"
var_expl_16S_pc <- round(100 * pcoa_res_16S_pc$values[[eig_col_16S_pc]][1:2], 1)

# Build plotting data
meta_16S_pc <- as(sample_data(phy_16S_pc), "data.frame") %>%
  rownames_to_column("sample_id")

pcoa_df_16S_pc <- as.data.frame(pcoa_res_16S_pc$vectors[, 1:2]) %>%
  rownames_to_column("sample_id") %>%
  rename(Axis1 = Axis.1, Axis2 = Axis.2) %>%
  left_join(meta_16S_pc, by = "sample_id") %>%
  mutate(group = recode(type, c = "Controls", p = "Patients"),
         group = factor(group, levels = c("Controls", "Patients")))

# p-value annotation
p_val_16S_pc   <- bc_test_16S_pc$p_value
p_label_16S_pc <- if (p_val_16S_pc < 0.001) "italic(p) < 0.001" else
                                            sprintf("italic(p) == %.3f", p_val_16S_pc)

# Plot
plot_pcoa_16S_pc <- ggplot(pcoa_df_16S_pc,
                           aes(x = Axis1, y = Axis2,
                               color = group, fill = group)) +
  stat_ellipse(geom = "polygon", alpha = 0.15,
               linewidth = 0.4, level = 0.95) +
  geom_point(size = 2) +
  scale_color_manual(values = group_colors_pc, name = NULL) +
  scale_fill_manual(values  = group_colors_pc, name = NULL) +
  labs(x = sprintf("PCoA 1 (%.1f%%)", var_expl_16S_pc[1]),
       y = sprintf("PCoA 2 (%.1f%%)", var_expl_16S_pc[2]),
       color = NULL, fill = NULL,
       title   = NULL,
       caption = NULL) +
  annotate("text", x = -Inf, y = Inf, label = p_label_16S_pc,
           hjust = -0, vjust = 1, parse = TRUE,
           size = 8 * 0.36) +
  theme_pub(8) +
  theme(legend.position  = "bottom",
        panel.grid.minor = element_blank(),
        plot.title       = element_text(face = "bold"),
        plot.caption     = element_text(hjust = 0, size = 7,
                                        margin = margin(t = 4)))

# Save (single-column journal width)
save_fig("./output_glioblastoma/pcoa_16S_pc.pdf",  plot_pcoa_16S_pc, width = 8.5, height = 8.5, units = "cm")
save_fig("./output_glioblastoma/pcoa_16S_pc.tiff", plot_pcoa_16S_pc, width = 8.5, height = 8.5, units = "cm",
       dpi = 600, compression = "lzw")

plot_pcoa_16S_pc

### Differential abundance

#### emuFit

In [ ]:
# Drop taxa with zero counts across all samples
filter_nonzero_taxa <- function(physeq) {
  keep <- taxa_sums(physeq) > 0
  prune_taxa(keep, physeq)
}

# Run emuFit 
fit_emu <- function(physeq, group_var) {
  taxa_to_test <- rownames(otu_table(physeq))

  emuFit(
    formula         = as.formula(paste0("~ ", group_var)),
    Y               = physeq,
    run_score_tests = TRUE,
    test_kj         = data.frame(k = 2, j = taxa_to_test),
    verbose         = TRUE
  )
}

In [ ]:
# Differential abundance: patients vs controls (16S genus, ITS species)

# Reference level = "c" (Controls) → estimate is log-fold-change Patients vs Controls
sample_data(phy_16S_gen_filtered)$type <-
  factor(sample_data(phy_16S_gen_filtered)$type, levels = c("c", "p"))
sample_data(phy_ITS_spc_filtered)$type <-
  factor(sample_data(phy_ITS_spc_filtered)$type, levels = c("c", "p"))

# Drop zero-only taxa
phy_16S_pc_emu <- filter_nonzero_taxa(phy_16S_gen_filtered)
phy_ITS_pc_emu <- filter_nonzero_taxa(phy_ITS_spc_filtered)

cat("Taxa retained (patients vs controls):\n",
    "  16S:", ntaxa(phy_16S_pc_emu), "\n",
    "  ITS:", ntaxa(phy_ITS_pc_emu), "\n")

# Fit
emu_fit_16S_pc <- fit_emu(phy_16S_pc_emu, group_var = "type")
emu_fit_ITS_pc <- fit_emu(phy_ITS_pc_emu, group_var = "type")



#### Tidy results

In [ ]:
tidy_emu <- function(fit, physeq, rank_col, prev_cutoff = 5) {

  # Multiple-testing correction settings
  adjust_method     <- "BH"
  sig_threshold_raw <- 0.05

  res <- fit$coef %>%
    dplyr::select(category, estimate, se, lower, upper, pval)

  tax <- as.data.frame(tax_table(physeq))
  tax$category <- rownames(tax)
  res <- dplyr::left_join(res, tax, by = "category")

  res <- res %>%
    dplyr::mutate(
      is_unresolved = grepl("_spc$| spc$|^NA( |_)?spc$|^NA$", .data[[rank_col]]) |
                      is.na(.data[[rank_col]]),
      label_raw = dplyr::case_when(
        is.na(.data[[rank_col]])                        ~ "unclassified",
        grepl("^NA( |_)?spc$|^NA$", .data[[rank_col]])  ~ "unclassified",
        grepl("_spc$| spc$", .data[[rank_col]])         ~ paste0(sub("( ?_?spc)+$", "", .data[[rank_col]]), " (unclassified)"),
        TRUE                                            ~ sub("^Candida glabrata$", "Nakaseomyces glabratus", as.character(.data[[rank_col]]))
      )
    ) %>%
    dplyr::group_by(label_raw) %>%
    dplyr::mutate(
      label = if (dplyr::n() > 1 && any(is_unresolved))
                paste0(label_raw, " ", dplyr::row_number())
              else label_raw
    ) %>%
    dplyr::ungroup()


  otu_counts <- otu_table(physeq)
  if (taxa_are_rows(physeq)) otu_counts <- t(otu_counts)
  non_zero <- data.frame(
    category              = colnames(otu_counts),
    non_zero_sample_count = as.integer(colSums(otu_counts > 0))
  )
  res <- dplyr::left_join(res, non_zero, by = "category")

  res <- res %>%
  dplyr::mutate(
      qval        = p.adjust(pval, method = adjust_method),
      significant = ifelse(pval <= sig_threshold_raw,
                           "significant (uncorrected)", "not significant"),
      color_group = dplyr::case_when(
        non_zero_sample_count < prev_cutoff & significant == "significant (uncorrected)" ~ "significant (uncorrected), few positive samples",
        non_zero_sample_count < prev_cutoff                                              ~ "not significant, few positive samples",
        significant == "significant (uncorrected)"                                       ~ "significant (uncorrected)",
        TRUE                                                                             ~ "not significant"
      ),
      label = factor(label, levels = unique(label[order(estimate)]))
    )
  res
}

In [ ]:
# Build tidy result tables
emu_16S_pc_res <- tidy_emu(emu_fit_16S_pc, phy_16S_pc_emu, rank_col = "genus")
emu_ITS_pc_res <- tidy_emu(emu_fit_ITS_pc, phy_ITS_pc_emu, rank_col = "species")

# Save
drop_cols <- c("significant", "color_group")

write.csv(dplyr::select(emu_16S_pc_res, -dplyr::any_of(drop_cols)),
          "./output_glioblastoma/radEmu_16S_PvC.csv", row.names = FALSE)
write.csv(dplyr::select(emu_ITS_pc_res, -dplyr::any_of(drop_cols)),
          "./output_glioblastoma/radEmu_ITS_PvC.csv", row.names = FALSE)

emu_ITS_pc_res  %>% arrange(pval)%>%head

#### Plot significantly differential abundant

In [ ]:
# Save helper: PDF + TIFF, height scales with taxa count
save_radEmu_plot <- function(plot, file_stub, n_taxa,
                             width_cm = 8.5, per_taxon_cm = 0.18, min_h = 8) {
  h <- max(min_h, n_taxa * per_taxon_cm)
  save_fig(paste0(file_stub, ".pdf"),  plot,
         width = width_cm, height = h, units = "cm")
  save_fig(paste0(file_stub, ".tiff"), plot,
         width = width_cm, height = h, units = "cm",
         dpi = 600, compression = "lzw", bg = "white")
}

In [ ]:
plot_radEmu_sig_only <- function(res, x_label,
                                 sig_col         = "black",
                                 sig_lowprev_col = "grey55",   # changed: colour for low-prevalence taxa
                                 base_size       = 7) {

  # changed: keep low-prevalence significant taxa instead of dropping them
  keep_groups <- c("significant (uncorrected)",
                   "significant (uncorrected), few positive samples")

  res_sig <- res %>%
    dplyr::filter(color_group %in% keep_groups) %>%
    dplyr::mutate(
      color_group = factor(color_group, levels = keep_groups),   
      label       = factor(label, levels = unique(label[order(estimate)]))
    )

  if (nrow(res_sig) == 0) {
    message("No significant taxa to plot.")
    return(invisible(NULL))
  }


  color_vals <- c(
    "significant (uncorrected)"                       = sig_col,
    "significant (uncorrected), few positive samples" = sig_lowprev_col
  )
  shape_vals <- c(
    "significant (uncorrected)"                       = 16,
    "significant (uncorrected), few positive samples" = 1
  )

  group_labs <- c(
    "significant (uncorrected)"                       = "\u2265 5 positive samples",
    "significant (uncorrected), few positive samples" = "< 5 positive samples"
  )

  ggplot(res_sig, aes(x = estimate, y = label,
                      color = color_group, shape = color_group)) +   
    geom_vline(xintercept = 0, linetype = "dashed", color = "grey40") +
    geom_errorbarh(aes(xmin = lower, xmax = upper), height = 0.3) +
    geom_point(size = 1.2) +
    scale_color_manual(values = color_vals, labels = group_labs, name = NULL) +
    scale_shape_manual(values = shape_vals, labels = group_labs, name = NULL) +  
    guides(colour = guide_legend(nrow = 1),
           shape  = guide_legend(nrow = 1)) +
    labs(x = x_label, y = NULL) +
    theme_pub(8) +
    theme(
      panel.grid.major.y = element_blank(),
      legend.position    = "bottom",
      legend.key.size    = unit(0.3, "cm"),
      axis.title.x       = element_text(size = 7),   
      axis.text          = element_text(size = 6.5, color ="black")  
    )
}

In [ ]:
p_16S_pc_sig <- plot_radEmu_sig_only(
  emu_16S_pc_res,
  x_label = "Log-fold-change (Patients vs Controls)"
)
p_ITS_pc_sig <- plot_radEmu_sig_only(
  emu_ITS_pc_res,
  x_label = "Log-fold-change (Patients vs Controls)"
)


n_sig <- function(res) sum(grepl("^significant", res$color_group))

n_16S_sig    <- n_sig(emu_16S_pc_res)
n_ITS_sig    <- n_sig(emu_ITS_pc_res)


save_radEmu_plot(p_16S_pc_sig, "./output_glioblastoma/radEmu_16S_PvC_sigOnly",
                 n_taxa = n_16S_sig)
save_radEmu_plot(p_ITS_pc_sig, "./output_glioblastoma/radEmu_ITS_PvC_sigOnly",
                 n_taxa = n_ITS_sig)

p_16S_pc_sig
p_ITS_pc_sig

## Patients compared by OS

In [ ]:
attach_OS <- function(ps, meta_OS) {
  ps_pat <- subset_samples(ps, type == "p")

  meta_existing <- as(sample_data(ps_pat), "data.frame")
  meta_existing$.rowid     <- rownames(meta_existing)
  meta_existing$patient_id <- str_sub(meta_existing$sample, -3, -1)   # "p001" -> "001"

  merged <- dplyr::left_join(
    meta_existing,
    meta_OS[, c("sample", "OS_group")],
    by = c("patient_id" = "sample")
  )
  rownames(merged) <- merged$.rowid
  merged$.rowid     <- NULL
  merged$patient_id <- NULL

  sample_data(ps_pat) <- sample_data(merged)
  ps_pat <- prune_samples(!is.na(sample_data(ps_pat)$OS_group), ps_pat)
  ps_pat <- prune_taxa(taxa_sums(ps_pat) > 0, ps_pat)
  ps_pat
}

phy_16S_gen_filtered_pat <- attach_OS(phy_16S_gen_filtered, complete_meta_patients_OS)
phy_ITS_spc_filtered_pat <- attach_OS(phy_ITS_spc_filtered, complete_meta_patients_OS)

sample_data(phy_16S_gen_filtered_pat)$OS_group <-
  factor(sample_data(phy_16S_gen_filtered_pat)$OS_group, levels = c("lowerOS", "higherOS"))
sample_data(phy_ITS_spc_filtered_pat)$OS_group <-
  factor(sample_data(phy_ITS_spc_filtered_pat)$OS_group, levels = c("lowerOS", "higherOS"))

table(sample_data(phy_16S_gen_filtered_pat)$OS_group)
table(sample_data(phy_ITS_spc_filtered_pat)$OS_group)

### Diversity

#### Alpha diversity

In [ ]:
### Diversity (Patients by OS)

# ITS Lower vs Higher OS, use S. cerevisiae as base (same base as for pc)
set.seed(123)
divnet_ITS_OS <- divnet(phy_ITS_spc_filtered_pat, X = "OS_group", base = "ASV2",
                        ncores = n_cores, tuning = "careful")



In [ ]:
# DivNet fits for Lower vs Higher OS
# - Genus level for 16S, species level for ITS
# - Default settings
# - X = "OS_group" uses the lowerOS/higherOS grouping in sample_data

# 16S Lower vs Higher OS
set.seed(123)
divnet_16S_OS <- divnet(phy_16S_gen_filtered_pat, X = "OS_group",
                        ncores = n_cores, tuning = "careful")

In [ ]:
#### Alpha diversity
testDiversity(divnet_ITS_OS, h0 = "shannon")
testDiversity(divnet_ITS_OS, h0 = "simpson")

In [ ]:

testDiversity(divnet_16S_OS, h0 = "shannon")
testDiversity(divnet_16S_OS, h0 = "simpson")

In [ ]:
# Run testDiversity for ITS and 16S, store p-values
td_its_shannon_OS <- testDiversity(divnet_ITS_OS, h0 = "shannon")
td_its_simpson_OS <- testDiversity(divnet_ITS_OS, h0 = "simpson")
td_16s_shannon_OS <- testDiversity(divnet_16S_OS, h0 = "shannon")
td_16s_simpson_OS <- testDiversity(divnet_16S_OS, h0 = "simpson")

pvals_its_OS <- c(
  Shannon = get_p_from_testdiv(td_its_shannon_OS),
  Simpson = get_p_from_testdiv(td_its_simpson_OS)
)
pvals_16s_OS <- c(
  Shannon = get_p_from_testdiv(td_16s_shannon_OS),
  Simpson = get_p_from_testdiv(td_16s_simpson_OS)
)

# Build summary tables and recode group labels
alpha_df_its_OS <- summarise_alpha_per_group(
  divnet_obj = divnet_ITS_OS,
  ps         = phy_ITS_spc_filtered_pat,
  group_var  = "OS_group"
) %>%
  mutate(group = recode(group, lowerOS = "Shorter OS", higherOS = "Longer OS"),
         group = factor(group, levels = c("Shorter OS", "Longer OS")))

alpha_df_16s_OS <- summarise_alpha_per_group(
  divnet_obj = divnet_16S_OS,
  ps         = phy_16S_gen_filtered_pat,
  group_var  = "OS_group"
) %>%
  mutate(group = recode(group, lowerOS = "Shorter OS", higherOS = "Longer OS"),
         group = factor(group, levels = c("Shorter OS", "Longer OS")))

# Group colors
group_colors_OS <- col_os   # central OS palette

# Build the plots — defaults tuned for half-page (single column) journal figures
plot_its_OS <- plot_alpha_grouped(
  alpha_df_its_OS,
  group_colors = group_colors_OS,
  pvals        = pvals_its_OS,
  title        = "ITS alpha diversity",
  caption      = NULL,
  base_size    = 8,
  point_size   = 2,
  errbar_w     = 0.12
)

plot_16s_OS <- plot_alpha_grouped(
  alpha_df_16s_OS,
  group_colors = group_colors_OS,
  pvals        = pvals_16s_OS,
  title        = "16S alpha diversity",
  caption      = NULL,
  base_size    = 8,
  point_size   = 2,
  errbar_w     = 0.12
)

plot_16s_OS <- plot_16s_OS + scale_y_continuous(expand = expansion(mult = c(0.05, 0.20)))
plot_its_OS <- plot_its_OS + scale_y_continuous(expand = expansion(mult = c(0.05, 0.20)))

# Save at single-column journal width (8.5 cm wide, ~6 cm tall)
save_fig("./output_glioblastoma/alpha_ITS_OS.pdf",  plot_its_OS, width = 8.5, height = 6, units = "cm")
save_fig("./output_glioblastoma/alpha_16S_OS.pdf",  plot_16s_OS, width = 8.5, height = 6, units = "cm")
# TIFF version at 600 dpi with LZW compression (journal standard)
save_fig("./output_glioblastoma/alpha_ITS_OS.tiff", plot_its_OS, width = 8.5, height = 6, units = "cm",
       dpi = 600, compression = "lzw")
save_fig("./output_glioblastoma/alpha_16S_OS.tiff", plot_16s_OS, width = 8.5, height = 6, units = "cm",
       dpi = 600, compression = "lzw")

plot_its_OS
plot_16s_OS

#### Bray-Curtis

In [ ]:
#### Bray-Curtis
# ITS
# construct matrix
n <- nsamples(phy_ITS_spc_filtered_pat)
X_diag <- diag(n)
colnames(X_diag) <- rownames(X_diag) <- as.character(1:n)

# calculate bray-curtis
set.seed(123)
divnet_ITS_OS_beta <- divnet(W = phy_ITS_spc_filtered_pat,
                             X = X_diag,
                             variance = "none", base = "ASV2")

# test bray-curtis
set.seed(123)
bc_test_ITS_OS <- testBetaDiversity(divnet_ITS_OS_beta,
                                    h0 = "bray-curtis",
                                    n_boot = 10000,
                                    sample_specimen_matrix = X_diag,
                                    groups = sample_data(phy_ITS_spc_filtered_pat)$OS_group)


bc_test_ITS_OS$p_value

In [ ]:
# 16S
# construct matrix
n <- nsamples(phy_16S_gen_filtered_pat)
X_diag <- diag(n)
colnames(X_diag) <- rownames(X_diag) <- as.character(1:n)

# calculate bray-curtis
set.seed(123)
divnet_16S_OS_beta <- divnet(W = phy_16S_gen_filtered_pat,
                             X = X_diag,
                             variance = "none")

# test bray-curtis
set.seed(123)
bc_test_16S_OS <- testBetaDiversity(divnet_16S_OS_beta,
                                    h0 = "bray-curtis",
                                    n_boot = 10000,
                                    sample_specimen_matrix = X_diag,
                                    groups = sample_data(phy_16S_gen_filtered_pat)$OS_group)

bc_test_16S_OS$p_value

In [ ]:
# Extract Bray-Curtis distance matrix from DivNet (ITS, lower vs higher OS)
bc_mat_ITS_OS <- divnet_ITS_OS_beta[["bray-curtis"]]

# Numerical noise on diagonal: clamp small negatives/values to 0
bc_mat_ITS_OS[bc_mat_ITS_OS < 0] <- 0
diag(bc_mat_ITS_OS) <- 0
bc_dist_ITS_OS <- as.dist(bc_mat_ITS_OS)

# PCoA (Cailliez correction handles negative eigenvalues from non-Euclidean BC)
pcoa_res_ITS_OS <- ape::pcoa(bc_dist_ITS_OS, correction = "cailliez")

# Variance explained — use corrected eigenvalues if Cailliez was applied
eig_col_ITS_OS  <- if ("Rel_corr_eig" %in% colnames(pcoa_res_ITS_OS$values))
                     "Rel_corr_eig" else "Relative_eig"
var_expl_ITS_OS <- round(100 * pcoa_res_ITS_OS$values[[eig_col_ITS_OS]][1:2], 1)

# Build plotting data
meta_ITS_OS <- as(sample_data(phy_ITS_spc_filtered_pat), "data.frame") %>%
  rownames_to_column("sample_id")

pcoa_df_ITS_OS <- as.data.frame(pcoa_res_ITS_OS$vectors[, 1:2]) %>%
  rownames_to_column("sample_id") %>%
  rename(Axis1 = Axis.1, Axis2 = Axis.2) %>%
  left_join(meta_ITS_OS, by = "sample_id") %>%
  mutate(group = recode(OS_group, lowerOS = "Shorter OS", higherOS = "Longer OS"),
         group = factor(group, levels = c("Shorter OS", "Longer OS")))

# p-value annotation
p_val_ITS_OS   <- bc_test_ITS_OS$p_value
p_label_ITS_OS <- if (p_val_ITS_OS < 0.001) "italic(p) < 0.001" else
                                            sprintf("italic(p) == %.3f", p_val_ITS_OS)

# Plot
plot_pcoa_ITS_OS <- ggplot(pcoa_df_ITS_OS,
                           aes(x = Axis1, y = Axis2,
                               color = group, fill = group)) +
  stat_ellipse(geom = "polygon", alpha = 0.15,
               linewidth = 0.4, level = 0.95) +
  geom_point(size = 2) +
  scale_color_manual(values = col_os, name = NULL) +
  scale_fill_manual(values  = col_os, name = NULL) +
  labs(x = sprintf("PCoA 1 (%.1f%%)", var_expl_ITS_OS[1]),
       y = sprintf("PCoA 2 (%.1f%%)", var_expl_ITS_OS[2]),
       color = NULL, fill = NULL,
       title   = NULL,
       caption = NULL) +
  annotate("text", x = -Inf, y = Inf, label = p_label_ITS_OS,
           hjust = -0, vjust = 1, parse = TRUE,
           size = 8 * 0.36) +
  theme_pub(8) +
  theme(legend.position  = "bottom",
        panel.grid.minor = element_blank(),
        plot.title       = element_text(face = "bold"),
        plot.caption     = element_text(hjust = 0, size = 7,
                                        margin = margin(t = 4)))

plot_pcoa_ITS_OS <- plot_pcoa_ITS_OS + scale_y_continuous(expand = expansion(mult = c(0.05, 0.20)))
                                      

# Save (single-column journal width)
save_fig("./output_glioblastoma/pcoa_ITS_OS.pdf",  plot_pcoa_ITS_OS, width = 8.5, height = 8.5, units = "cm")
save_fig("./output_glioblastoma/pcoa_ITS_OS.tiff", plot_pcoa_ITS_OS, width = 8.5, height = 8.5, units = "cm",
       dpi = 600, compression = "lzw")

plot_pcoa_ITS_OS

In [ ]:
# Extract Bray-Curtis distance matrix from DivNet (16S, lower vs higher OS)
bc_mat_16S_OS <- divnet_16S_OS_beta[["bray-curtis"]]

# Numerical noise on diagonal: clamp small negatives/values to 0
bc_mat_16S_OS[bc_mat_16S_OS < 0] <- 0
diag(bc_mat_16S_OS) <- 0
bc_dist_16S_OS <- as.dist(bc_mat_16S_OS)

# PCoA (Cailliez correction handles negative eigenvalues from non-Euclidean BC)
pcoa_res_16S_OS <- ape::pcoa(bc_dist_16S_OS, correction = "cailliez")

# Variance explained — use corrected eigenvalues if Cailliez was applied
eig_col_16S_OS  <- if ("Rel_corr_eig" %in% colnames(pcoa_res_16S_OS$values))
                     "Rel_corr_eig" else "Relative_eig"
var_expl_16S_OS <- round(100 * pcoa_res_16S_OS$values[[eig_col_16S_OS]][1:2], 1)

# Build plotting data
meta_16S_OS <- as(sample_data(phy_16S_gen_filtered_pat), "data.frame") %>%
  rownames_to_column("sample_id")

pcoa_df_16S_OS <- as.data.frame(pcoa_res_16S_OS$vectors[, 1:2]) %>%
  rownames_to_column("sample_id") %>%
  rename(Axis1 = Axis.1, Axis2 = Axis.2) %>%
  left_join(meta_16S_OS, by = "sample_id") %>%
  mutate(group = recode(OS_group, lowerOS = "Shorter OS", higherOS = "Longer OS"),
         group = factor(group, levels = c("Shorter OS", "Longer OS")))

# p-value annotation
p_val_16S_OS   <- bc_test_16S_OS$p_value
p_label_16S_OS <- if (p_val_16S_OS < 0.001) "italic(p) < 0.001" else
                                            sprintf("italic(p) == %.3f", p_val_16S_OS)

# Plot
plot_pcoa_16S_OS <- ggplot(pcoa_df_16S_OS,
                           aes(x = Axis1, y = Axis2,
                               color = group, fill = group)) +
  stat_ellipse(geom = "polygon", alpha = 0.15,
               linewidth = 0.4, level = 0.95) +
  geom_point(size = 2) +
  scale_color_manual(values = col_os, name = NULL) +
  scale_fill_manual(values  = col_os, name = NULL) +
  labs(x = sprintf("PCoA 1 (%.1f%%)", var_expl_16S_OS[1]),
       y = sprintf("PCoA 2 (%.1f%%)", var_expl_16S_OS[2]),
       color = NULL, fill = NULL,
       title   = NULL,
       caption = NULL) +
  annotate("text", x = -Inf, y = Inf, label = p_label_16S_OS,
           hjust = -0, vjust = 1, parse = TRUE,
           size = 8 * 0.36) +
  theme_pub(8) +
  theme(legend.position  = "bottom",
        panel.grid.minor = element_blank(),
        plot.title       = element_text(face = "bold"),
        plot.caption     = element_text(hjust = 0, size = 7,
                                        margin = margin(t = 4)))

plot_pcoa_16S_OS <- plot_pcoa_16S_OS + scale_y_continuous(expand = expansion(mult = c(0.05, 0.20)))                                        

# Save (single-column journal width)
save_fig("./output_glioblastoma/pcoa_16S_OS.pdf",  plot_pcoa_16S_OS, width = 8.5, height = 8.5, units = "cm")
save_fig("./output_glioblastoma/pcoa_16S_OS.tiff", plot_pcoa_16S_OS, width = 8.5, height = 8.5, units = "cm",
       dpi = 600, compression = "lzw")

plot_pcoa_16S_OS

### Barplots

In [ ]:
tbl_16S_rel_gen_filtered <- phy_16S_gen_filtered %>%
  transform_sample_counts(function(x) x / sum(x)) %>% psmelt() %>% as_tibble()
tbl_ITS_rel_filtered <- phy_ITS_spc_filtered %>%
  transform_sample_counts(function(x) x / sum(x)) %>% psmelt() %>% as_tibble()

In [ ]:
tbl_ITS_rel_filtered %>%
  group_by(species) %>% summarise(m = mean(Abundance)) %>%
  arrange(desc(m)) %>% head(11)

In [ ]:
tbl_16S_rel_gen_filtered %>%
  group_by(genus) %>% summarise(m = mean(Abundance)) %>%
  arrange(desc(m)) %>% head(11)

In [ ]:
# Sort samples by bray-curtis
order_samples_by_bray <- function(ps, group_var = "plot_group", id_col = "sample",
                                  group_levels = c("Control", "Shorter OS", "Longer OS")) {
  ps_rel <- transform_sample_counts(ps, function(x) x / sum(x))
  otu <- as(otu_table(ps_rel), "matrix")
  if (taxa_are_rows(ps_rel)) otu <- t(otu)
  md  <- as(sample_data(ps_rel), "data.frame")
  ids <- md[[id_col]]; names(ids) <- rownames(md)
  grp <- md[[group_var]]
  unlist(lapply(group_levels, function(g) {
    rn <- rownames(md)[!is.na(grp) & grp == g]
    if (length(rn) <= 2) return(unname(ids[rn]))
    d <- vegan::vegdist(otu[rn, , drop = FALSE], method = "bray")
    unname(ids[rn[hclust(d, method = "average")$order]])
  }), use.names = FALSE)
}

# distinguish to groups controls and patients higher and lower OS
add_plot_group <- function(ps, ps_pat) {
  os_by_name <- setNames(as.character(sample_data(ps_pat)$OS_group), sample_names(ps_pat))
  if (length(os_by_name) == 0 || all(is.na(os_by_name)))
    stop("ps_pat contains no info on OS.")
  md <- as(sample_data(ps), "data.frame")
  message("add_plot_group: ", sum(!is.na(os_by_name[rownames(md)])),
          " Patients with OS, ", sum(md$type == "c"), " Controls")
  sample_data(ps)$plot_group <- dplyr::case_when(
    md$type == "c"                         ~ "Control",
    os_by_name[rownames(md)] == "lowerOS"  ~ "Shorter OS",
    os_by_name[rownames(md)] == "higherOS" ~ "Longer OS",
    TRUE                                   ~ NA_character_
  )
  ps
}

# Barplot (16S/ITS identical)
make_os_barplot <- function(tbl_long, ps_pg, order_ids, rank_col, fill_title) {
  grp_map <- as(sample_data(ps_pg), "data.frame")[, c("sample", "plot_group")]
  df <- tbl_long %>%
    dplyr::left_join(grp_map, by = "sample") %>%
    dplyr::filter(!is.na(plot_group)) %>%
    dplyr::mutate(
      taxon = {
        raw <- dplyr::case_when(
          is.na(.data[[rank_col]])                        ~ NA_character_,
          grepl("^NA( |_)?spc$|^NA$", .data[[rank_col]])  ~ NA_character_,
          grepl("(_| )spc$", .data[[rank_col]])           ~ sub("( ?_?spc)+$", " (unclassified)", .data[[rank_col]]),
          TRUE                                            ~ as.character(.data[[rank_col]])
        )
        forcats::fct_na_value_to_level(forcats::fct_lump(raw, n = 10, w = Abundance), "Other")
      },
      sample = factor(sample, levels = order_ids),
      super  = factor(ifelse(plot_group == "Control", "Controls", "Patients"),
                      levels = c("Controls", "Patients")),
      os_sub = factor(dplyr::case_when(plot_group == "Control"    ~ " ",
                                       plot_group == "Shorter OS" ~ "Shorter OS",
                                       TRUE                       ~ "Longer OS"),
                      levels = c(" ", "Shorter OS", "Longer OS"))
    )

  # build palette from the actual levels, force "Other" to grey
  lv   <- levels(df$taxon)
  vals <- setNames(expandedPalette(length(lv)), lv)
  vals["Other"] <- "grey80"

  ggplot(df, aes(x = sample, y = Abundance, fill = taxon)) +
    geom_bar(stat = "identity", position = "fill") +
    ggh4x::facet_nested(~ super + os_sub, scales = "free_x", space = "free_x") +
    scale_fill_manual(values = vals, name = fill_title) +
    scale_x_discrete(labels = NULL, expand = c(0, 0)) +
    scale_y_continuous(limits = c(0, 1), expand = c(0, 0), labels = scales::percent) +
    labs(x = NULL, y = "Relative abundance") +
    theme(axis.ticks.x = element_blank())   # theme_pub() is global
}



In [ ]:
# ============ PATIENT/CONTROL-BARPLOTS (OS-Split, BC-sorted) ===============

phy_16S_gen_filtered_pat <- attach_OS(phy_16S_gen_filtered, complete_meta_patients_OS)
phy_ITS_spc_filtered_pat <- attach_OS(phy_ITS_spc_filtered, complete_meta_patients_OS)
sample_data(phy_16S_gen_filtered_pat)$OS_group <-
  factor(sample_data(phy_16S_gen_filtered_pat)$OS_group, levels = c("lowerOS", "higherOS"))
sample_data(phy_ITS_spc_filtered_pat)$OS_group <-
  factor(sample_data(phy_ITS_spc_filtered_pat)$OS_group, levels = c("lowerOS", "higherOS"))

# --- 16S (Genus) ---
phy_16S_pg <- add_plot_group(phy_16S_gen_filtered, phy_16S_gen_filtered_pat)
order_16S  <- order_samples_by_bray(phy_16S_pg)
p_bar_16S  <- make_os_barplot(tbl_16S_rel_gen_filtered, phy_16S_pg, order_16S, "genus", "Genus")
save_fig("./output_glioblastoma/barplot_16S_ctrl_OS_bray.tiff", p_bar_16S,
         width = W_DOUBLE, height = 8, units = "cm", dpi = 300, compression = "lzw")
p_bar_16S

# --- ITS (Species) ---
phy_ITS_pg <- add_plot_group(phy_ITS_spc_filtered, phy_ITS_spc_filtered_pat)
order_ITS  <- order_samples_by_bray(phy_ITS_pg)
p_bar_ITS  <- make_os_barplot(tbl_ITS_rel_filtered, phy_ITS_pg, order_ITS, "species", "Species")
save_fig("./output_glioblastoma/barplot_ITS_ctrl_OS_bray.tiff", p_bar_ITS,
         width = W_DOUBLE, height = 8, units = "cm", dpi = 300, compression = "lzw")
p_bar_ITS


In [ ]:
# neg controls

neg_16S <- c("341F-785R-P01-F06-Extcontrol-empty", "341F-785R-P01-H06-Buffer-empty")
neg_ITS <- c("fITS7-ITS4-P01-F06-Extcontrol-empty", "fITS7-ITS4-P01-H06-Buffer-empty")

plot_neg_controls <- function(ps_raw, neg_ids, rank_col, marker) {
  ps <- prune_samples(sample_names(ps_raw) %in% neg_ids, ps_raw)
  ps <- prune_taxa(taxa_sums(ps) > 0, ps)
  reads  <- sample_sums(ps)
  ps_rel <- transform_sample_counts(ps, function(x) x / sum(x))
  df <- psmelt(ps_rel)
  df$ctrl  <- ifelse(grepl("Extcontrol", df$Sample, ignore.case = TRUE), "Extraction", "Buffer")
  df$xlab  <- paste0(df$ctrl, "\n(", reads[as.character(df$Sample)], " reads)")
  df$taxon <- forcats::fct_lump(
                ifelse(is.na(df[[rank_col]]), "Unclassified", as.character(df[[rank_col]])),
                n = 10, w = df$Abundance)
  ggplot(df, aes(x = xlab, y = Abundance, fill = taxon)) +
    geom_bar(stat = "identity", position = "fill") +
    scale_y_continuous(limits = c(0, 1), expand = c(0, 0), labels = scales::percent) +
    scale_x_discrete(expand = c(0, 0)) +
    labs(x = NULL, y = "Relative abundance",
         title = paste0(marker, " negative controls"), fill = rank_col) +
    theme_pub(8) + theme(axis.ticks.x = element_blank())
}

p_neg_16S <- plot_neg_controls(phy_16S_raw, neg_16S, "genus",   "16S")
p_neg_ITS <- plot_neg_controls(phy_ITS_raw, neg_ITS, "species", "ITS")

p_neg_16S
p_neg_ITS

### Differential abundance

#### emuFit

In [ ]:
# Differential abundance: lowerOS vs higherOS (patients only)

# Reference level = "lowerOS" → estimate is log-fold-change higherOS vs lowerOS
sample_data(phy_16S_gen_filtered_pat)$OS_group <-
  factor(sample_data(phy_16S_gen_filtered_pat)$OS_group, levels = c("lowerOS", "higherOS"))
sample_data(phy_ITS_spc_filtered_pat)$OS_group <-
  factor(sample_data(phy_ITS_spc_filtered_pat)$OS_group, levels = c("lowerOS", "higherOS"))

# Drop zero-only taxa
phy_16S_OS_emu <- filter_nonzero_taxa(phy_16S_gen_filtered_pat)
phy_ITS_OS_emu <- filter_nonzero_taxa(phy_ITS_spc_filtered_pat)

cat("Taxa retained (OS groups):\n",
    "  16S:", ntaxa(phy_16S_OS_emu), "\n",
    "  ITS:", ntaxa(phy_ITS_OS_emu), "\n")

# Fit (slow - run once)
emu_fit_16S_OS <- fit_emu(phy_16S_OS_emu, group_var = "OS_group")
emu_fit_ITS_OS <- fit_emu(phy_ITS_OS_emu, group_var = "OS_group")


#### Tidy results

In [ ]:
# Build tidy result tables
emu_16S_OS_res <- tidy_emu(emu_fit_16S_OS, phy_16S_OS_emu, rank_col = "genus")
emu_ITS_OS_res <- tidy_emu(emu_fit_ITS_OS, phy_ITS_OS_emu, rank_col = "species")

# Save
drop_cols <- c("significant", "color_group")

write.csv(dplyr::select(emu_16S_OS_res, -dplyr::any_of(drop_cols)),
          "./output_glioblastoma/radEmu_16S_OS.csv", row.names = FALSE)
write.csv(dplyr::select(emu_ITS_OS_res, -dplyr::any_of(drop_cols)),
          "./output_glioblastoma/radEmu_ITS_OS.csv", row.names = FALSE)

#### Plot significantly differential abundant

In [ ]:
p_16S_OS_sig <- plot_radEmu_sig_only(
  emu_16S_OS_res,
    x_label = "Log-fold-change (Longer OS vs Shorter OS)"
)
p_ITS_OS_sig <- plot_radEmu_sig_only(
  emu_ITS_OS_res,
    x_label = "Log-fold-change (Longer OS vs Shorter OS)"
)



n_16S_OS_sig <- n_sig(emu_16S_OS_res)
n_ITS_OS_sig <- n_sig(emu_ITS_OS_res)


save_radEmu_plot(p_16S_OS_sig, "./output_glioblastoma/radEmu_16S_OS_sigOnly",
                 n_taxa = n_16S_OS_sig)
save_radEmu_plot(p_ITS_OS_sig, "./output_glioblastoma/radEmu_ITS_OS_sigOnly",
                 n_taxa = n_ITS_OS_sig)

p_16S_OS_sig
p_ITS_OS_sig

# Composite Figures

## Main figures

### Figure 2

In [ ]:
# helper

tag_style   <- theme(plot.tag = element_text(face = "bold", size = 9))
add_tag     <- function(p, tag) p + labs(tag = tag) + tag_style

panel_style <- theme(plot.title = element_text(face = "bold", size = 8, hjust = 0),
                     strip.text = element_text(face = "plain", size = 7))

bar_legend  <- theme(legend.position = "right",
                     legend.key.size = unit(2.5, "mm"),
                     legend.text     = element_text(size = 5.5),
                     legend.title    = element_text(size = 6))

# Alpha diversity for a single marker, one facet per index
plot_alpha_marker <- function(alpha_df, pvals, cols, title) {
  fmt_p <- function(p) if (p < 0.001) "italic(p) < 0.001" else sprintf("italic(p) == %.3f", p)

  pval_df <- alpha_df %>%
    dplyr::group_by(index) %>%
    dplyr::summarise(y = max(upper) + 0.15 * diff(range(c(lower, upper))), .groups = "drop") %>%
    dplyr::mutate(label = vapply(pvals[index], fmt_p, character(1)))

  ggplot(alpha_df, aes(group, estimate, color = group)) +
    geom_point(size = 2) +
    geom_errorbar(aes(ymin = lower, ymax = upper), width = 0.12, linewidth = 0.5) +
    geom_text(data = pval_df, aes(x = 1.5, y = y, label = label),
              inherit.aes = FALSE, parse = TRUE, size = 2.4) +
    facet_wrap(~ index, scales = "free_y") +
    scale_y_continuous(expand = expansion(mult = c(0.08, 0.22))) +
    scale_color_manual(values = cols) +
    labs(x = NULL, y = "Estimate", title = title) +
    theme_pub(8) + theme(legend.position = "none")
}

In [ ]:
p2A <- add_tag(p_bar_16S + labs(x = NULL, title = "16S rRNA (genus)") + bar_legend + panel_style, "A")
p2B <- add_tag(p_bar_ITS + labs(x = NULL, title = "ITS (species)")    + bar_legend + panel_style, "B")
p2C <- add_tag(plot_alpha_marker(alpha_df_16s, pvals_16s, col_pvc,
                                 "16S rRNA: Alpha diversity") +
                 guides(colour = "none") + panel_style, "C")

p2D <- add_tag(plot_alpha_marker(alpha_df_its, pvals_its, col_pvc,
                                 "ITS: Alpha diversity") +
                 guides(colour = "none") + panel_style, "D")
p2E <- add_tag(plot_pcoa_16S_pc + labs(title = "16S rRNA: Bray–Curtis PCoA") + panel_style, "E")
p2F <- add_tag(plot_pcoa_ITS_pc + labs(title = "ITS: Bray–Curtis PCoA") + panel_style, "F")

# Top row: alpha diversity, bottom row: Bray-Curtis PCoA (16S left, ITS right)
p2_div <- ((p2C | p2D) / (p2E | p2F)) +
  plot_layout(widths = c(1, 1), guides = "collect") &
  theme(legend.position    = "bottom",
        legend.box.spacing = unit(1, "mm"))

p_fig2 <- (p2A / p2B / p2_div) +
  plot_layout(heights = c(0.8, 0.8, 2.2))

save_fig("./output_glioblastoma/Fig2_communities_PvC.pdf",  p_fig2,
         width = W_DOUBLE, height = 23, units = "cm")
save_fig("./output_glioblastoma/Fig2_communities_PvC.tiff", p_fig2,
         width = W_DOUBLE, height = 23, units = "cm",
         dpi = 600, compression = "lzw")

p_fig2

### Figure 3

In [ ]:
p3A <- add_tag(p_16S_pc_sig +
                 labs(x = NULL, title = "16S rRNA (genus)") +
                 panel_style +
                 guides(colour = "none", shape = "none"), "A")   # legend only in B

p3B <- add_tag(p_ITS_pc_sig + labs(title = "ITS (species)") + panel_style +
                 guides(colour = guide_legend(nrow = 2, byrow = TRUE),
                        shape  = guide_legend(nrow = 2, byrow = TRUE)), "B")                 

p_fig3 <- (p3A / p3B) +
  plot_layout(heights = c(n_16S_sig + 3, n_ITS_sig + 3), guides = "collect") &
  theme(legend.position = "bottom",
        legend.key.size = unit(0.3, "cm"),
        legend.margin   = margin(t = -2))

save_fig("./output_glioblastoma/Fig3_radEmu_PvC.pdf",  p_fig3,
         width = W_SINGLE, height = 15, units = "cm")
save_fig("./output_glioblastoma/Fig3_radEmu_PvC.tiff", p_fig3,
         width = W_SINGLE, height = 15, units = "cm",
         dpi = 600, compression = "lzw")

p_fig3

### Figure 4

In [ ]:
add_tag_block <- function(p, tag) wrap_elements(full = p) + labs(tag = tag) + tag_style

In [ ]:

# Reads PNG or TIFF, always returns a [height, width, channel] array
read_img <- function(path) {
  path <- as.character(path)                 # str_glue returns a glue object
  if (!file.exists(path)) stop("File not found: ", path)

  magic   <- as.integer(readBin(path, "raw", n = 4))
  is_png  <- identical(magic[1:4], c(137L, 80L, 78L, 71L))
  is_tiff <- identical(magic[1:2], c(73L, 73L)) || identical(magic[1:2], c(77L, 77L))

  img <- if (is_tiff)     tiff::readTIFF(path, native = FALSE)
         else if (is_png) png::readPNG(path)
         else stop("Unsupported format: ", path)

  if (is.list(img))
    stop("Multi-frame image: convert to RGB in Fiji (Image > Type > RGB Color).")
  if (length(dim(img)) == 2)                                  # greyscale
    img <- array(rep(img, 3), dim = c(dim(img), 3))
  if (dim(img)[3] == 4) img <- img[, , 1:3, drop = FALSE]     # drop alpha
  img
}


# Crop by relative coordinates; x and y run 0-1 with (0,0) at bottom left
crop_img <- function(img, x = c(0, 1), y = c(0, 1)) {
  h <- dim(img)[1]; w <- dim(img)[2]
  clamp <- function(v, n) pmin(pmax(round(v), 1), n)
  r1 <- clamp((1 - y[2]) * h, h); r2 <- clamp((1 - y[1]) * h, h)
  c1 <- clamp(x[1] * w, w);       c2 <- clamp(x[2] * w, w)
  img[r1:r2, c1:c2, , drop = FALSE]
}

# Image as a ggplot panel; scale bar drawn only when um_per_px is supplied
img_panel <- function(img, title = NULL, um_per_px = NULL, bar_um = 50,
                      bar_col = "white") {
  h <- dim(img)[1]; w <- dim(img)[2]

  p <- ggplot(data.frame(x = 0:1, y = 0:1), aes(x, y)) +
    geom_blank() +
    annotation_custom(rasterGrob(img, interpolate = TRUE), 0, 1, 0, 1) +
    coord_fixed(ratio = h / w, xlim = c(0, 1), ylim = c(0, 1), expand = FALSE) +
    labs(title = title) +
    theme_void() + panel_style

  if (!is.null(um_per_px)) {
    frac <- bar_um / (um_per_px * w)          # bar length relative to cropped width
    p <- p +
      annotate("rect", xmin = 0.95 - frac, xmax = 0.95,
               ymin = 0.05, ymax = 0.075, fill = bar_col) +
      annotate("text", x = 0.95 - frac / 2, y = 0.10,
               label = paste0(bar_um, " \u00b5m"),
               colour = bar_col, size = 2, vjust = 0)
  }
  p
}

In [ ]:
img <- read_img(str_glue("{path_meta}iba1_case_composite.tif"))

In [ ]:
UM <- 0.6818185

if_chans  <- c("dapi", "gfap", "iba1", "composite")
if_titles <- c("DAPI", "GFAP", "Iba1", "Merge")

# Scale bar only on the last panel to avoid repeating it four times
if_panels <- purrr::pmap(
  list(if_chans, if_titles, seq_along(if_chans)),
  function(ch, ti, i) {
    img_panel(read_img(str_glue("{path_meta}iba1_case_{ch}.tif")), ti,
              um_per_px = if (i == length(if_chans)) UM else NULL)
  })


In [ ]:
# --- Figure 4: systemic and intratumoral immune profile and survival ---

p4A <- add_tag_block(
  wrap_plots(plot_list_il, ncol = 4) + guide_area() +
    plot_layout(guides = "collect") &
    scale_fill_manual(values = col_pvc_raw,
                      labels = pvc_labels) &
    theme(legend.position = "bottom",
          legend.key.size = unit(3, "mm"),
          legend.text     = element_text(size = 6),
        axis.text.x     = element_blank(),
        axis.ticks.x    = element_blank()),
  "A")

UM      <- 0.65                              # pixel width from Fiji: Image > Properties
img_mrg <- read_img(str_glue("{path_meta}iba1_case_composite.tif"))


# Detail window: lower left quadrant, 0.45 edge length
box_x <- c(0.05, 0.50); box_y <- c(0.05, 0.50)

# B: overview with the detail window outlined
p4B <- add_tag(
  img_panel(img_mrg, "Overview", um_per_px = UM, bar_um = 100) +
    annotate("rect", xmin = box_x[1], xmax = box_x[2],
             ymin = box_y[1], ymax = box_y[2],
             fill = NA, colour = "white", linewidth = 0.4),
  "B")

# C: detail
p4C <- add_tag(
  img_panel(crop_img(img_mrg, box_x, box_y), "Detail",
            um_per_px = UM, bar_um = 50),
  "C")

# D: Kaplan-Meier
p4D <- add_tag(p_km_all$plot + theme_pub(8) + panel_style +
                 theme(legend.position = "none") +
                 labs(title = "Overall survival"), "D")


# E: multivariable Cox model, full width
p4E <- add_tag_block(p_cox_forest_final, "E")

# Absolute widths for the two image cells so no whitespace shifts the tags
p_mid <- (p4B | p4C | p4D) +
  plot_layout(widths = unit(c(5.0, 5.0, 1), c("cm", "cm", "null")))

p_fig4 <- (p4A / p_mid / p4E) +
  plot_layout(heights = unit(c(1, 4.0, 1.2), c("null", "cm", "null")))

save_fig("./output_glioblastoma/Fig4_immune_survival.pdf",  p_fig4,
         width = W_DOUBLE, height = 21, units = "cm")
save_fig("./output_glioblastoma/Fig4_immune_survival.tiff", p_fig4,
         width = W_DOUBLE, height = 21, units = "cm",
         dpi = 600, compression = "lzw")

p_fig4

## Supplementary figures

### Figure S1

In [ ]:
# --- Supplementary Figure S1: diversity by overall survival group ---

pS1A <- add_tag(plot_alpha_marker(alpha_df_16s_OS, pvals_16s_OS, col_os,
                                  "16S rRNA: Alpha diversity") +
                  guides(colour = "none"), "A")

pS1B <- add_tag(plot_alpha_marker(alpha_df_its_OS, pvals_its_OS, col_os,
                                  "ITS: Alpha diversity") +
                  guides(colour = "none"), "B")

pS1C <- add_tag(plot_pcoa_16S_OS +
                  labs(title = "16S rRNA: Bray–Curtis PCoA") + panel_style, "C")

pS1D <- add_tag(plot_pcoa_ITS_OS +
                  labs(title = "ITS: Bray–Curtis PCoA") + panel_style, "D")


pS1 <- ((pS1A | pS1B) / (pS1C | pS1D)) +
  plot_layout(widths = c(1, 1), guides = "collect") &
  theme(legend.position = "bottom")

save_fig("./output_glioblastoma/FigS1_diversity_OS.pdf",  pS1,
         width = W_DOUBLE, height = 12, units = "cm")
save_fig("./output_glioblastoma/FigS1_diversity_OS.tiff", pS1,
         width = W_DOUBLE, height = 12, units = "cm",
         dpi = 600, compression = "lzw")

pS1

### Figure S2

In [ ]:
# --- Supplementary Figure S2: differentially abundant taxa, lower vs. higher OS ---

pS2A <- add_tag(p_16S_OS_sig + labs(x = NULL, title = "16S rRNA (genus)") +
                  panel_style + guides(colour = "none", shape = "none"), "A")
pS2B <- add_tag(p_ITS_OS_sig + labs(title = "ITS (species)") + panel_style, "B")

pS2 <- (pS2A / pS2B) +
  plot_layout(heights = c(n_16S_OS_sig + 3, n_ITS_OS_sig + 3), guides = "collect") &
  theme(legend.position = "bottom",
        legend.key.size = unit(0.3, "cm"),
        legend.margin   = margin(t = -2))

save_fig("./output_glioblastoma/FigS2_radEmu_OS.pdf",  pS2,
         width = W_SINGLE, height = 9, units = "cm")
save_fig("./output_glioblastoma/FigS2_radEmu_OS.tiff", pS2,
         width = W_SINGLE, height = 9, units = "cm",
         dpi = 600, compression = "lzw")

pS2

### Figure S3

In [ ]:
# --- Supplementary Figure S3: systemic cytokines by overall survival group ---

pS3 <- wrap_plots(plot_list_il_os, ncol = 4) + guide_area() +
  plot_layout(guides = "collect") &
  scale_fill_manual(values = col_os_raw,
                                        labels = os_labels) &
  theme(legend.position = "bottom",
        legend.key.size = unit(3, "mm"),
        legend.text     = element_text(size = 6))

save_fig("./output_glioblastoma/FigS3_cytokines_OS.pdf",  pS3,
         width = W_DOUBLE, height = 9, units = "cm")
save_fig("./output_glioblastoma/FigS3_cytokines_OS.tiff", pS3,
         width = W_DOUBLE, height = 9, units = "cm",
         dpi = 600, compression = "lzw")

pS3

# Save Data

In [ ]:
#save Rdata
save.image(file = "./output_glioblastoma/Glioblastoma_code_for_publication.RData")

In [ ]:
# Export sample group assignments needed to reproduce the microbiome analyses
mk_mapping <- function(ps_full, ps_pat, marker) {
  md <- as(sample_data(ps_full), "data.frame")
  os <- setNames(as.character(sample_data(ps_pat)$OS_group), sample_names(ps_pat))
  data.frame(
    library_name = rownames(md),        # sequencing library name as delivered by the provider
    participant  = md$sample,           # corresponds to the ENA sample alias
    marker       = marker,
    type         = md$type,               # patient vs control
    OS_group     = unname(os[rownames(md)]),
    row.names    = NULL
  )
}

sample_mapping <- rbind(
  mk_mapping(phy_16S_gen_filtered, phy_16S_gen_filtered_pat, "16S"),
  mk_mapping(phy_ITS_spc_filtered, phy_ITS_spc_filtered_pat, "ITS")
)


write.csv(sample_mapping, "./output_glioblastoma/sample_mapping.csv", row.names = FALSE)
sample_mapping

In [ ]:
writeLines(capture.output(sessionInfo()),
           "./output_glioblastoma/sessionInfo.txt")